# Pipeline Local PostgreSQL + pgvector - Corpus Completo

Este notebook executa um pipeline local para construir uma base juridica em PostgreSQL com pgvector. O fluxo captura leis em fonte oficial, preserva o inteiro teor, gera chunks juridicos, grava os dados em tabelas relacionais e cria uma camada vetorial consultavel por similaridade.

Modo desta versao: `corpus_completo`.

Esta versao usa a mesma logica da amostra, preparada para processar o corpus de leis federais de 1988 a 2026 com checkpoint e lotes.

O notebook foi organizado para que cada etapa produza uma saida verificavel antes da proxima etapa consumir esses dados. Isso reduz ambiguidade em execucoes longas e torna possivel localizar rapidamente se uma falha ocorreu na descoberta, na captura, no tratamento textual, na carga relacional ou na camada vetorial.

Contratos principais do pipeline:

- metadados obtidos nos Dados Abertos;
- inteiro teor capturado em fonte oficial;
- texto normalizado com preservacao do conteudo normativo;
- chunks juridicos rastreaveis por URN, hash e ordem;
- persistencia em PostgreSQL com extensao pgvector;
- validacoes executaveis apos cada etapa critica;
- interrupcao explicita quando houver lacuna de cobertura.


## 1. Configuracao operacional

Define escopo de execucao, caminhos de trabalho, parametros de coleta, modelo de embeddings e conexao PostgreSQL. O mesmo bloco atende execucao local e Colab, sem bifurcar a logica do pipeline.

A amostra descobre as leis no periodo completo pelos Dados Abertos e usa `SAMPLE_LIMIT` para controlar quantas serao processadas. Para fixar um conjunto especifico, informe `SAMPLE_URNS` como lista JSON.

O modelo padrao e `intfloat/multilingual-e5-small`, compativel com o schema `vector(384)` e com o formato E5 usado nas consultas. Em Colab limpo, o modelo e resolvido por nome e mantido em cache local. Para execucao offline ou com artefato fechado, configure `EMBEDDING_MODEL_PATH` para uma pasta de modelo ja disponivel no ambiente.

A escolha do modelo fica explicita, mas nao fica travada no codigo. O notebook registra uma lista de modelos de referencia para comparacao posterior. Modelos com dimensao diferente de 384 exigem ajuste coordenado de `EMBEDDING_DIMENSION`, schema `vector(...)`, tabela ou banco de destino; por isso o notebook de entrega mantem um padrao unico e validavel.

A pergunta de verificacao da busca tambem fica parametrizada em `PERGUNTA_TESTE_BUSCA`. O valor padrao foi escolhido para dialogar com a norma de referencia da amostra e evitar uma demonstracao operacional com termos desconectados do conjunto pequeno de teste. No corpus completo, a mesma pergunta continua valida como consulta semantica aberta.

Saidas criadas nesta etapa:

- diretorios de dados brutos, processados, lotes, relatorios e vetores;
- parametros de coleta e retomada;
- modelo, dimensao, consulta de teste e alternativas documentadas;
- mapa centralizado de caminhos usados pelas etapas seguintes.


In [ ]:
import html
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


# Dependencias instaladas aqui sao as minimas para iniciar o notebook.
# Bibliotecas pesadas ou ligadas ao banco ficam nas etapas em que sao usadas.
def garantir_pacote(pacote: str, import_name: str | None = None) -> None:
    nome_import = import_name or pacote
    if importlib.util.find_spec(nome_import) is None:
        print(f"Instalando dependencia: {pacote}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])


garantir_pacote("requests")
import requests


EH_COLAB = "google.colab" in sys.modules
CWD = Path.cwd()

NOTEBOOK_MODE = "corpus_completo"
RODAR_CORPUS_COMPLETO = True


# Permite trocar a amostra por variavel de ambiente sem editar o restante do fluxo.
# Aceita JSON, texto unico ou lista separada por virgula/quebra de linha.
def parse_sample_urns(value: str) -> list[str]:
    value = (value or "").strip()
    if not value:
        return []
    try:
        parsed = json.loads(value)
        if isinstance(parsed, str):
            return [parsed.strip()] if parsed.strip() else []
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except json.JSONDecodeError:
        pass
    return [part.strip() for part in re.split(r"[,;\n]+", value) if part.strip()]


SAMPLE_URNS = parse_sample_urns(os.environ.get("SAMPLE_URNS", ""))
SAMPLE_LIMIT = int(os.environ.get("SAMPLE_LIMIT", "20"))
URN_REFERENCIA_VALIDACAO = os.environ.get(
    "URN_REFERENCIA_VALIDACAO",
    "urn:lex:br:federal:lei:2026-05-11;15407",
).strip()

ANO_INICIO = 2026
ANO_FIM = 1988
TIPO_NORMA_SENADO = "LEI"

if EH_COLAB:
    BASE_DIR = Path("/content") / "corpus_completo_pgvector_local"
else:
    BASE_DIR = CWD / "corpus_completo_pgvector_local"

# Cada grupo de artefatos fica isolado para facilitar auditoria e reexecucao.
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SHARDS_DIR = DATA_DIR / "shards"
LOCAL_EXPORT_PACKAGE_DIR = DATA_DIR / "pacote_jsonl_local"
BATCHES_DIR = DATA_DIR / "batches"
VECTORS_DIR = DATA_DIR / "vectors"
REPORTS_DIR = DATA_DIR / "reports"
CHECKPOINTS_DIR = DATA_DIR / "checkpoints"
LOGS_DIR = DATA_DIR / "logs"

for path in [RAW_DIR, PROCESSED_DIR, SHARDS_DIR, LOCAL_EXPORT_PACKAGE_DIR, BATCHES_DIR, VECTORS_DIR, REPORTS_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = "pipeline-pgvector-local-v1"

# A amostra e o completo usam a mesma descoberta. O limite altera volume, nao rota.
LIMITE_COLETA = None if RODAR_CORPUS_COMPLETO else SAMPLE_LIMIT
LIMITE_COLETA_AMOSTRA = LIMITE_COLETA

# A amostra limpa saidas por padrao; o completo preserva checkpoint para retomada.
RESETAR_ARQUIVOS_PROCESSADOS = os.environ.get(
    "RESETAR_ARQUIVOS_PROCESSADOS",
    "false",
).strip().lower() in {"1", "true", "sim", "yes"}
TAMANHO_LOTE_MANIFEST = 100
TAMANHO_SHARD_NORMAS = 100
TAMANHO_SHARD_CHUNKS = 2000

PAUSA_SENADO_SEGUNDOS = 0.05
PAUSA_PLANALTO_SEGUNDOS = 0.08
TIMEOUT_HTTP = 30

EXECUTAR_PUBLICACAO_REMOTA = False

# O Colab limpo nao possui pasta de modelo; por padrao resolvemos por nome.
# Para execucao offline/reprodutivel, informe EMBEDDING_MODEL_PATH explicitamente.
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "intfloat/multilingual-e5-small").strip()
EMBEDDING_MODEL_PATH = os.environ.get("EMBEDDING_MODEL_PATH", "").strip()
EMBEDDING_MODEL_SOURCE = "local_path" if EMBEDDING_MODEL_PATH else "model_id"
EMBEDDING_MODEL_LABEL = os.environ.get(
    "EMBEDDING_MODEL_LABEL",
    Path(EMBEDDING_MODEL_PATH).name if EMBEDDING_MODEL_PATH else EMBEDDING_MODEL,
).strip()
EMBEDDING_LOCAL_FILES_ONLY = os.environ.get(
    "EMBEDDING_LOCAL_FILES_ONLY",
    "false",
).strip().lower() in {"1", "true", "sim", "yes"}
EMBEDDING_CACHE_DIR = os.environ.get(
    "EMBEDDING_CACHE_DIR",
    str(BASE_DIR / "models" / "sentence_transformers"),
).strip()
MODELOS_EMBEDDING_REFERENCIA = [
    {
        "modelo": "intfloat/multilingual-e5-small",
        "dimensao": 384,
        "uso": "padrao do notebook; multilingue, leve e compativel com vector(384)",
    },
    {
        "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "dimensao": 384,
        "uso": "alternativa multilingue leve para comparacao local",
    },
    {
        "modelo": "sentence-transformers/distiluse-base-multilingual-cased-v2",
        "dimensao": 512,
        "uso": "alternativa multilingue; exige ajustar EMBEDDING_DIMENSION e schema",
    },
    {
        "modelo": "BAAI/bge-m3",
        "dimensao": 1024,
        "uso": "alternativa mais pesada; exige ajustar EMBEDDING_DIMENSION e schema",
    },
]
EMBEDDING_DIMENSION = int(os.environ.get("EMBEDDING_DIMENSION", "384"))
EMBEDDING_BATCH_SIZE = int(os.environ.get("EMBEDDING_BATCH_SIZE", "32"))
EMBEDDING_NORMALIZE = True
EMBEDDING_INPUT_FORMAT = "e5-prefix-v1"
PERGUNTA_TESTE_BUSCA = os.environ.get(
    "PERGUNTA_TESTE_BUSCA",
    "regime disciplinar diferenciado em estabelecimento penal federal",
).strip()

# Os nomes seguem variaveis comuns de PostgreSQL para facilitar execucao local e Colab.
DB_HOST = os.environ.get("POSTGRES_HOST", "127.0.0.1")
DB_PORT = int(os.environ.get("POSTGRES_PORT", "5432"))
DB_NAME = os.environ.get("POSTGRES_DB", "juridico_pgvector")
DB_USER = os.environ.get("POSTGRES_USER", "postgres")
DB_PASSWORD = os.environ.get("POSTGRES_PASSWORD", "juridico_pgvector_local")
DB_CONTAINER_NAME = os.environ.get("POSTGRES_CONTAINER_NAME", "juridico_pgvector_notebook")
PREPARAR_POSTGRES_COLAB = True
USAR_DOCKER_LOCAL = not EH_COLAB
RESETAR_TABELAS = False

URN_VALIDACAO_TEXTO_INTEGRAL = URN_REFERENCIA_VALIDACAO

PATHS = {
    "manifest": RAW_DIR / "manifest_leis.jsonl",
    "manifest_summary": PROCESSED_DIR / "manifest_summary.json",
    "checkpoint": CHECKPOINTS_DIR / "coleta_checkpoint.json",
    "normas": PROCESSED_DIR / "normas_integras.jsonl",
    "chunks": PROCESSED_DIR / "chunks_juridicos.jsonl",
    "falhas": LOGS_DIR / "falhas_coleta.jsonl",
    "validacao_json": REPORTS_DIR / "relatorio_validacao.json",
    "validacao_md": REPORTS_DIR / "relatorio_validacao.md",
    "cobertura_json": REPORTS_DIR / "relatorio_cobertura_coleta.json",
    "auditoria_fina_json": REPORTS_DIR / "auditoria_padrao_textual.json",
    "auditoria_fina_md": REPORTS_DIR / "auditoria_padrao_textual.md",
    "embeddings": VECTORS_DIR / "chunk_embeddings_pgvector_local.jsonl",
    "embedding_summary": VECTORS_DIR / "embedding_summary.json",
    "pgvector_summary": REPORTS_DIR / "pgvector_summary.json",
}

if EMBEDDING_CACHE_DIR:
    Path(EMBEDDING_CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "modo": NOTEBOOK_MODE,
    "base_dir": str(BASE_DIR),
    "rodar_corpus_completo": RODAR_CORPUS_COMPLETO,
    "sample_urns": SAMPLE_URNS,
    "sample_limit": SAMPLE_LIMIT,
    "limite_coleta": LIMITE_COLETA,
    "resetar_arquivos_processados": RESETAR_ARQUIVOS_PROCESSADOS,
    "urn_referencia_validacao": URN_REFERENCIA_VALIDACAO,
    "periodo": f"{ANO_FIM}-{ANO_INICIO}",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_model_path": EMBEDDING_MODEL_PATH or None,
    "embedding_model_label": EMBEDDING_MODEL_LABEL,
    "embedding_model_source": EMBEDDING_MODEL_SOURCE,
    "embedding_local_files_only": EMBEDDING_LOCAL_FILES_ONLY,
    "embedding_cache_dir": EMBEDDING_CACHE_DIR or None,
    "modelos_embedding_referencia": MODELOS_EMBEDDING_REFERENCIA,
    "pergunta_teste_busca": PERGUNTA_TESTE_BUSCA,
    "db_host": DB_HOST,
    "db_port": DB_PORT,
    "db_name": DB_NAME,
    "db_user": DB_USER,
    "resetar_tabelas": RESETAR_TABELAS,
    "publicacao_remota": EXECUTAR_PUBLICACAO_REMOTA,
}, ensure_ascii=False, indent=2))


def texto_console(texto: str) -> str:
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    return str(texto).encode(encoding, errors="replace").decode(encoding, errors="replace")


{
  "modo": "corpus_completo",
  "base_dir": "/content/corpus_completo_pgvector_local",
  "rodar_corpus_completo": true,
  "sample_urns": [],
  "sample_limit": 20,
  "limite_coleta": null,
  "resetar_arquivos_processados": false,
  "urn_referencia_validacao": "urn:lex:br:federal:lei:2026-05-11;15407",
  "periodo": "1988-2026",
  "embedding_model": "intfloat/multilingual-e5-small",
  "embedding_model_path": null,
  "embedding_model_label": "intfloat/multilingual-e5-small",
  "embedding_model_source": "model_id",
  "embedding_local_files_only": false,
  "embedding_cache_dir": "/content/corpus_completo_pgvector_local/models/sentence_transformers",
  "modelos_embedding_referencia": [
    {
      "modelo": "intfloat/multilingual-e5-small",
      "dimensao": 384,
      "uso": "padrao do notebook; multilingue, leve e compativel com vector(384)"
    },
    {
      "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "dimensao": 384,
      "uso": "alternativa multiling

### Checagem da configuracao

Valida diretorios, periodo, limite de coleta, dimensao vetorial e parametros minimos de banco antes de iniciar chamadas externas. Esta checagem evita iniciar requisicoes ou escrita em disco quando a configuracao basica esta inconsistente.


In [ ]:
assert ANO_INICIO >= ANO_FIM
assert DATA_DIR.exists()
assert RAW_DIR.exists()
assert PROCESSED_DIR.exists()
assert CHECKPOINTS_DIR.exists()
assert RODAR_CORPUS_COMPLETO or SAMPLE_LIMIT > 0
assert LIMITE_COLETA is None or LIMITE_COLETA == SAMPLE_LIMIT
assert EMBEDDING_MODEL or EMBEDDING_MODEL_PATH
assert EMBEDDING_DIMENSION == 384
assert PERGUNTA_TESTE_BUSCA
assert DB_NAME
assert DB_USER
print("Configuracao aprovada.")


Configuracao aprovada.


## 2. Biblioteca interna do pipeline

Reune as funcoes de IO, requisicao, descoberta, enriquecimento de metadados, resolucao de fonte oficial, limpeza, chunking, checkpoint e validacao textual. A biblioteca fica embutida para manter o notebook autocontido.

A ordem interna segue a dependencia real do processo: primeiro utilitarios e HTTP, depois metadados oficiais, em seguida resolucao do inteiro teor, normalizacao conservadora, montagem canonica, geracao de chunks e validacoes. As funcoes desta celula sao usadas tanto pela amostra quanto pelo corpus completo.


In [ ]:
# Biblioteca interna do pipeline.
# Organizacao:
# 1. utilitarios de arquivo e HTTP;
# 2. descoberta e detalhamento de leis nos Dados Abertos;
# 3. resolucao de URLs oficiais a partir da URN;
# 4. extracao, limpeza e acabamento do inteiro teor;
# 5. montagem do registro canonico, chunking e validacoes.

SESSION = requests.Session()
SESSION.headers.update(
    {
        "Accept": "application/json, text/html;q=0.9, */*;q=0.8",
        "User-Agent": "Mozilla/5.0 juridico-pgvector-local",
    }
)
SENADO_BASE_URL = "https://legis.senado.leg.br/dadosabertos/legislacao"

def as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

def sha256_text(text: str) -> str:
    import hashlib
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def request_json(url: str, params: dict[str, str] | None = None, tentativas: int = 3) -> dict[str, Any]:
    ultimo_erro: Exception | None = None
    for tentativa in range(1, tentativas + 1):
        try:
            response = SESSION.get(url, params=params, timeout=TIMEOUT_HTTP)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            ultimo_erro = exc
            time.sleep(min(2 * tentativa, 6))
    raise RuntimeError(f"Falha ao consultar JSON {url}: {ultimo_erro}")

def decodificar_response_text(response: requests.Response) -> str:
    content = response.content
    if content.startswith((b"\xff\xfe", b"\xfe\xff")):
        return content.decode("utf-16", errors="replace")
    amostra = content[:2000]
    if amostra and amostra.count(b"\x00") > len(amostra) // 8:
        for encoding in ("utf-16", "utf-16-le", "utf-16-be"):
            try:
                return content.decode(encoding)
            except UnicodeDecodeError:
                continue
    encoding = response.encoding or response.apparent_encoding or "windows-1252"
    text = content.decode(encoding, errors="replace")
    if "\ufffd" in text:
        text = content.decode("windows-1252", errors="replace")
    return text

def request_text(url: str, tentativas: int = 3) -> tuple[str, str]:
    ultimo_erro: Exception | None = None
    for tentativa in range(1, tentativas + 1):
        try:
            response = SESSION.get(url, timeout=TIMEOUT_HTTP, allow_redirects=True)
            response.raise_for_status()
            return decodificar_response_text(response), str(response.url)
        except Exception as exc:
            ultimo_erro = exc
            time.sleep(min(2 * tentativa, 6))
    raise RuntimeError(f"Falha ao baixar HTML {url}: {ultimo_erro}")

def parse_data_br(data: str | None) -> str | None:
    if not data:
        return None
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(data, fmt).date().isoformat()
        except ValueError:
            pass
    return None

def normalizar_numero(numero: Any) -> str:
    return str(numero or "").strip().replace(".", "")

def urn_from_norma(norm: str | None) -> str | None:
    if not norm:
        return None
    match = re.match(r"LEI-(\d+)-(\d{4})-(\d{2})-(\d{2})", norm)
    if not match:
        return None
    numero, ano, mes, dia = match.groups()
    return f"urn:lex:br:federal:lei:{ano}-{mes}-{dia};{numero}"

def urn_from_url(url: str | None) -> str | None:
    if not url:
        return None
    match = re.search(r"urn:lex:br:[^\"'&\s]+", url)
    return match.group(0) if match else None

def listar_leis_por_ano(ano: int, pausa_segundos: float = PAUSA_SENADO_SEGUNDOS) -> list[dict[str, Any]]:
    data = request_json(f"{SENADO_BASE_URL}/lista", params={"tipo": TIPO_NORMA_SENADO, "ano": str(ano)})
    documentos = data.get("ListaDocumento", {}).get("documentos", {}).get("documento")
    leis: list[dict[str, Any]] = []
    for documento in as_list(documentos):
        numero = normalizar_numero(documento.get("numero"))
        data_assinatura = parse_data_br(documento.get("dataassinatura"))
        leis.append(
            {
                "tipo_norma": "LEI",
                "numero": numero,
                "ano": str(documento.get("anoassinatura") or ano),
                "data_assinatura": data_assinatura,
                "norma": documento.get("norma"),
                "norma_nome": documento.get("normaNome"),
                "ementa": documento.get("ementa"),
                "urn": urn_from_norma(documento.get("norma")),
                "senado_id": str(documento.get("id") or ""),
                "status": "discovered",
                "tentativas": 0,
                "ultimo_erro": None,
            }
        )
    if pausa_segundos:
        time.sleep(pausa_segundos)
    return leis

def descobrir_leis_intervalo(ano_inicio: int = ANO_INICIO, ano_fim: int = ANO_FIM) -> list[dict[str, Any]]:
    leis: list[dict[str, Any]] = []
    for ano in range(ano_inicio, ano_fim - 1, -1):
        print(f"Descobrindo leis de {ano}...")
        leis.extend(listar_leis_por_ano(ano))
    return sorted(
        leis,
        key=lambda item: (
            int(item.get("ano") or 0),
            item.get("data_assinatura") or "",
            int(item.get("numero") or 0),
        ),
        reverse=True,
    )

def consultar_detalhe_lei(numero: str, ano: str, tipo: str = "LEI") -> dict[str, Any]:
    data = request_json(f"{SENADO_BASE_URL}/{tipo}/{numero}/{ano}.json")
    documento = data.get("DetalheDocumento", {}).get("documentos", {}).get("documento")
    docs = as_list(documento)
    if not docs:
        raise RuntimeError(f"Detalhe nao encontrado para {tipo} {numero}/{ano}.")
    doc = docs[0]
    identificacao = doc.get("identificacao", {})
    url_documento = identificacao.get("urlDocumento")
    return {
        "tipo_norma": "LEI",
        "numero": normalizar_numero(identificacao.get("numero") or numero),
        "ano": str(ano),
        "data_assinatura": parse_data_br(identificacao.get("dataassinatura")),
        "norma": identificacao.get("norma"),
        "norma_nome": identificacao.get("normaNome"),
        "ementa": doc.get("ementa") or identificacao.get("ementa"),
        "urn": urn_from_url(url_documento) or urn_from_norma(identificacao.get("norma")),
        "url_documento": url_documento,
        "senado_id": str(doc.get("id") or ""),
        "raw_senado": doc,
    }

def limpar_texto(texto: str) -> str:
    import html
    texto = html.unescape(texto or "")
    texto = texto.replace("\ufeff", " ").replace("\xa0", " ")
    texto = re.sub(r"\r", "\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r" *\n *", "\n", texto)
    texto = re.sub(r"(?m)^([IVXLCDM]+\s*[-–])\s*\n\s*(\()", r"\1 \2", texto)
    texto = re.sub(r"(\([^)\n]*)\n([^)]*\))", r"\1 \2", texto)
    texto = re.sub(r"\bArt\.\s*([0-9]+)\s+o\b", r"Art. \1º", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

def remover_blocos_html_nao_textuais(html_text: str) -> str:
    texto = re.sub(r"(?is)<script.*?</script>", " ", html_text)
    texto = re.sub(r"(?is)<style.*?</style>", " ", texto)
    texto = re.sub(r"(?is)<!--.*?-->", " ", texto)
    return texto

def html_generico_para_texto(html_text: str, inicio: int, fim: int | None = None) -> str:
    trecho = html_text[inicio:fim] if fim is not None else html_text[inicio:]
    trecho = remover_blocos_html_nao_textuais(trecho)
    trecho = re.sub(r"(?i)<br\s*/?>", "\n", trecho)
    trecho = re.sub(r"(?i)</(p|div|center|h1|h2|h3|li|tr)>", "\n", trecho)
    trecho = re.sub(r"(?i)<(p|div|center|h1|h2|h3|li|tr)(?:\s[^>]*)?>", "\n", trecho)
    trecho = re.sub(r"(?is)<[^>]+>", " ", trecho)
    return limpar_texto(trecho)

HTML_SPACE = r"(?:\s|&nbsp;|&#160;|\xa0)*"
CABECALHO_NORMA = re.compile(
    rf"(LEI\s+(COMPLEMENTAR\s+)?N{HTML_SPACE}"
    rf"(?:[ºo°]|<sup>{HTML_SPACE}(?:<u>)?{HTML_SPACE}o{HTML_SPACE}(?:</u>)?{HTML_SPACE}</sup>)?{HTML_SPACE}"
    rf"|DECRETO(?:\s*-\s*LEI)?\s+N{HTML_SPACE}"
    rf"(?:[ºo°]|<sup>{HTML_SPACE}(?:<u>)?{HTML_SPACE}o{HTML_SPACE}(?:</u>)?{HTML_SPACE}</sup>)?{HTML_SPACE}"
    rf"|MEDIDA\s+PROVIS[OÓ]RIA\s+N{HTML_SPACE}"
    rf"(?:[ºo°]|<sup>{HTML_SPACE}(?:<u>)?{HTML_SPACE}o{HTML_SPACE}(?:</u>)?{HTML_SPACE}</sup>)?{HTML_SPACE})"
    r"\d",
    re.I | re.S,
)
FIM_TEXTO_PLANALTO = re.compile(r"Este texto n[aã]o substitui", re.I)

def html_para_texto_normativo_generico(html_text: str) -> str:
    inicio = html_text.find('<div class="textoNorma">')
    if inicio != -1:
        fim_match = FIM_TEXTO_PLANALTO.search(html_text, inicio)
        fim = fim_match.start() if fim_match else None
        rodape = html_text.find('<div class="rodapeTexto">', inicio)
        if rodape != -1:
            fim = rodape if fim is None else min(rodape, fim)
        inicio_texto = inicio
        h1_matches = list(re.finditer(r"(?is)<h1[^>]*>.*?</h1>", html_text[:inicio]))
        if h1_matches and CABECALHO_NORMA.search(h1_matches[-1].group(0)):
            inicio_texto = h1_matches[-1].start()
        return html_generico_para_texto(html_text, inicio_texto, fim)

    body_match = re.search(r"<body[^>]*>", html_text, re.I)
    inicio_busca = body_match.end() if body_match else 0
    titulo_match = CABECALHO_NORMA.search(html_text, inicio_busca)
    if titulo_match:
        inicio = titulo_match.start()
        fim_match = FIM_TEXTO_PLANALTO.search(html_text, inicio)
        fim = fim_match.start() if fim_match else None
        return html_generico_para_texto(html_text, inicio, fim)
    raise RuntimeError("Nao foi possivel localizar o inicio do texto normativo no HTML.")

def polir_texto_normativo(texto: str) -> str:
    texto = limpar_texto(texto)
    texto = re.sub(r"\bD\ufffd([A\u00c1]gua)\b", r"D'\1", texto, flags=re.I)
    texto = re.sub(r"\b(ADENDO)\s+\ufffd([A-Z])", r'\1 "\2', texto)
    texto = re.sub(r"\s+\ufffd\s+", " - ", texto)
    texto = re.sub(r"\bN\s+o\b", "N\u00ba", texto)
    texto = re.sub(r"\bn\s+o\b", "n\u00ba", texto)
    texto = re.sub(
        r"(?m)^(LEI(?:\s+COMPLEMENTAR)?\s+N[\u00ba\u00b0o]?\s*[0-9][^\n]*,)"
        r"\s*\n+\s*(DE\s+\d{1,2}[\u00ba\u00b0]?\s+DE\s+[A-Z\u00c7][^\n]+)",
        r"\1 \2",
        texto,
    )
    texto = re.sub(
        r"(?m)^(LEI(?:\s+COMPLEMENTAR)?\s+N[\u00ba\u00b0o]?\s*[0-9][^\n]*,)"
        r"\s*\n+\s*DE\s+(\d{1,2}[\u00ba\u00b0]?)\s+([A-Z\u00c7][^\n]+)",
        r"\1 DE \2 DE \3",
        texto,
    )
    texto = re.sub(
        r"(?m)^(LEI(?:\s+COMPLEMENTAR)?\s+N[\u00ba\u00b0o]?\s*[0-9][^\n]*,\s*DE\s+\d{1,2}[\u00ba\u00b0]?)"
        r"\s*\n+\s*(DE\s+[A-Z\u00c7][^\n]+)",
        r"\1 \2",
        texto,
    )
    texto = re.sub(r"(?m)^(LEI)\s*\n+\s*(N[º°o]\s*[^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+COMPLEMENTAR)\s*\n+\s*(N[º°o]\s*[^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+N[º°o]?)\s*\n+\s*([0-9][^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+COMPLEMENTAR\s+N[º°o]?)\s*\n+\s*([0-9][^\n]+)", r"\1 \2", texto)
    texto = re.sub(
        r"(?m)^(LEI(?:\s+COMPLEMENTAR)?\s+N[º°o]?\s*[0-9][^\n]*,\s*DE)"
        r"\s*\n+\s*(\d{1,2})\s*\n+\s*DE\s*\n+\s*([A-ZÇ][^\n]+)",
        r"\1 \2 DE \3",
        texto,
    )
    texto = re.sub(r"(?m)^(LEI\s+N[º°o]?\s*[0-9][^\n]*,\s*DE)\s*\n+\s*([0-9][^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+COMPLEMENTAR\s+N[º°o]?\s*[0-9][^\n]*,\s*DE)\s*\n+\s*([0-9][^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+N[º°o]?\s*[0-9][^\n]*,\s*DE\s+\d{1,2}\s+DE)\s*\n+\s*([A-ZÇ][^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+COMPLEMENTAR\s+N[º°o]?\s*[0-9][^\n]*,\s*DE\s+\d{1,2}\s+DE)\s*\n+\s*([A-ZÇ][^\n]+)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+N[º°o]?\s*[0-9][^\n]*,\s*DE[^\n]*\bDE)\s*\n+\s*(\d{4}\.?)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^(LEI\s+COMPLEMENTAR\s+N[º°o]?\s*[0-9][^\n]*,\s*DE[^\n]*\bDE)\s*\n+\s*(\d{4}\.?)", r"\1 \2", texto)
    texto = re.sub(r"(?im)^(Fa[çc]o saber[^\n]*\ba)\s*\n+\s*(seguinte Lei:)", r"\1 \2", texto)
    texto = re.sub(r"(?m)^Art[ \t]*\n+\s*([0-9]+(?:[-–][A-Z])?[º°]?)", r"Art. \1", texto)
    texto = re.sub(r"(?m)^Art[ \t]+([0-9]+(?:[-–][A-Z])?[º°]?)", r"Art. \1", texto)
    texto = re.sub(r"\bArt\s+\.", "Art.", texto)
    texto = re.sub(r"\bArt\.\s*\n+\s*([0-9]+)\s*\n+\s*([º°])", r"Art. \1\2", texto)
    texto = re.sub(r"\bArt\.\s*\n+\s*([0-9]+(?:[-–][A-Z])?[º°]?)", r"Art. \1", texto)
    texto = re.sub(
        r"(?m)^(Art\.\s*[0-9]+(?:[-–][A-Z])?[º°]?)[ \t]*\n+(?:[ \t]*\n+)*[ \t]*"
        r"(?!Art\.|T[ÍI]TULO|CAP[ÍI]TULO|Se[çc][aã]o|Subse[çc][aã]o)([^\n]+)",
        r"\1 \2",
        texto,
    )
    texto = re.sub(r"(?m)^Art\.[ \t]*\n+(?:[ \t]*\n+)*[ \t]*(\((?:Revogad[oa]|VETADO)[^\n]*\))", r"Art. \1", texto)
    texto = re.sub(r"(?im)^(Fa[çc]o)\s*\n+\s*(saber\b)", r"\1 \2", texto)
    texto = re.sub(r"\b([0-9]+)\s+o\b", lambda match: f"{match.group(1)}º", texto)
    texto = re.sub(r" +([,.;:])", r"\1", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

MESES_PT_BR = {
    "01": "JANEIRO", "02": "FEVEREIRO", "03": "MARÇO", "04": "ABRIL",
    "05": "MAIO", "06": "JUNHO", "07": "JULHO", "08": "AGOSTO",
    "09": "SETEMBRO", "10": "OUTUBRO", "11": "NOVEMBRO", "12": "DEZEMBRO",
}
LINHAS_RUIDO_TOPO = re.compile(
    r"^(Mensagem(?:\s+de\s+veto)?|Produ[çc][aã]o\s+de\s+efeito|"
    r"Partes\s+mantidas|Vig[êé]ncia|Regulamento|Texto\s+compilado|"
    r"Convers[ãa]o\s+da\s+Medida|Convers[ãa]o\s+da\s+MP|"
    r"\(Vide\b.*\)|\(Revogado\b.*\))",
    re.I,
)
INICIO_PREAMBULO_RE = re.compile(r"^(O\s+PRESIDENTE|A\s+PRESIDENTA|Fa[çc]o\s+saber)\b", re.I)
INICIO_ARTIGO_RE = re.compile(r"^(Art\.|Artigo\s+[UuÚú]nico\b)")
INICIO_HIERARQUIA_RE = re.compile(r"^(T[ÍI]TULO|CAP[ÍI]TULO|Se[çc][aã]o|Subse[çc][aã]o)\b", re.I)
INICIO_BLOCO_RE = re.compile(
    r"^(Art\.|Artigo\s+[UuÚú]nico\b|Par[áa]grafo\s+[UuÚú]nico\.|"
    r"§\s*\d+|[IVXLCDM]+\s*[-–]|[a-z]\)|"
    r"T[ÍI]TULO\b|CAP[ÍI]TULO\b|Se[çc][aã]o\b|Subse[çc][aã]o\b)",
    re.I,
)

FECHO_FORMAL_RE = re.compile(
    r"\bBras[ií]lia,\s+\d{1,2}\s+de\s+[a-zç]+(?:\s+de\s+\d{4})?"
    r"(?:;\s*[^.\n]*?(?:Rep[uú]blica\.|$)|\.)?",
    re.I,
)

def linhas_nao_vazias(texto: str) -> list[str]:
    return [linha.strip() for linha in texto.splitlines() if linha.strip()]

def numero_norma_oficial(numero: str) -> str:
    numero_limpo = re.sub(r"\D", "", numero or "")
    if len(numero_limpo) <= 3:
        return numero_limpo
    return f"{numero_limpo[:-3]}.{numero_limpo[-3:]}"

def titulo_canonico_por_urn(urn: str) -> str | None:
    info = extrair_info_urn(urn or "")
    if not info:
        return None
    ano, mes, dia = info.data_norma.split("-")
    tipo = "LEI COMPLEMENTAR" if info.tipo_norma == "lei.complementar" else "LEI"
    dia_formatado = "1º" if dia == "01" else str(int(dia))
    return f"{tipo} Nº {numero_norma_oficial(info.numero)}, DE {dia_formatado} DE {MESES_PT_BR.get(mes, mes)} DE {ano}"

def linha_ruido_topo(linha: str) -> bool:
    return bool(LINHAS_RUIDO_TOPO.match(linha.strip()))

def linha_inicia_bloco_juridico(linha: str) -> bool:
    return bool(INICIO_BLOCO_RE.match(linha)) and not re.match(r"^art\.", linha)

def normalizar_ementa(ementa: str | None) -> str | None:
    texto = limpar_texto(ementa or "")
    texto = re.sub(r"\s+", " ", texto).strip().rstrip(".")
    if not texto or linha_ruido_topo(texto) or re.fullmatch(r"Disp[oõ]e", texto, re.I):
        return None
    return texto

def normalizar_linha_hierarquia(linha: str) -> str:
    linha = linha.strip()
    if not INICIO_HIERARQUIA_RE.match(linha):
        return linha
    linha = re.sub(r"(?i)^t[ií]tulo\b", "TÍTULO", linha)
    linha = re.sub(r"(?i)^cap[ií]tulo\b", "CAPÍTULO", linha)
    linha = re.sub(r"(?i)^se[cç][aã]o\b", "Seção", linha)
    linha = re.sub(r"(?i)^subse[cç][aã]o\b", "Subseção", linha)
    linha = re.sub(r"\b[UuÚú]nico\b", "ÚNICO", linha)
    return linha

def remover_nota_publicacao_consolidada(linhas: list[str]) -> list[str]:
    resultado: list[str] = []
    pulando = False
    for linha in linhas:
        if re.match(r"^PUBLICA[ÇC][ÃA]O\s+CONSOLIDADA\b", linha, re.I):
            pulando = True
            continue
        if pulando:
            if INICIO_PREAMBULO_RE.match(linha) or INICIO_HIERARQUIA_RE.match(linha) or INICIO_ARTIGO_RE.match(linha):
                pulando = False
            else:
                continue
        resultado.append(linha)
    return resultado

def extrair_ementa_topo(linhas: list[str], inicio: int) -> tuple[str | None, int]:
    partes: list[str] = []
    idx = inicio
    while idx < len(linhas):
        linha = linhas[idx]
        if linha_ruido_topo(linha) or re.match(r"^PUBLICA[ÇC][ÃA]O\b", linha, re.I):
            idx += 1
            continue
        if INICIO_PREAMBULO_RE.match(linha) or INICIO_HIERARQUIA_RE.match(linha) or INICIO_ARTIGO_RE.match(linha):
            break
        partes.append(linha)
        idx += 1
    return normalizar_ementa(" ".join(partes)) if partes else None, idx

def extrair_preambulo(linhas: list[str], inicio: int) -> tuple[str | None, int]:
    idx = inicio
    while idx < len(linhas) and not INICIO_PREAMBULO_RE.match(linhas[idx]):
        idx += 1
    if idx >= len(linhas):
        return None, inicio
    partes: list[str] = []
    while idx < len(linhas):
        linha = linhas[idx]
        if partes and (INICIO_HIERARQUIA_RE.match(linha) or INICIO_ARTIGO_RE.match(linha)):
            break
        partes.append(linha)
        idx += 1
        if re.search(r"seguinte\s+lei\s*:", linha, re.I):
            break
    preambulo = re.sub(r"\s+", " ", limpar_texto(" ".join(partes))).strip()
    return preambulo, idx

def formatar_blocos_corpo(linhas: list[str]) -> list[str]:
    blocos: list[str] = []
    for linha_original in linhas:
        linha = normalizar_linha_hierarquia(linha_original)
        if linha_ruido_topo(linha):
            continue
        if linha_inicia_bloco_juridico(linha) or not blocos:
            blocos.append(linha)
            continue
        blocos[-1] = f"{blocos[-1]} {linha}".strip()
    formatados: list[str] = []
    for bloco in blocos:
        if INICIO_HIERARQUIA_RE.match(bloco) and formatados and formatados[-1] != "" and not INICIO_HIERARQUIA_RE.match(formatados[-1]):
            formatados.append("")
        formatados.append(bloco)
    return formatados

def token_assinatura_caixa_alta(token: str) -> bool:
    token_limpo = re.sub(r"[^A-Za-zÀ-ÿ]", "", token or "")
    return bool(token_limpo) and token_limpo.upper() == token_limpo and len(token_limpo) > 1

def separar_assinaturas(texto: str) -> list[str]:
    texto = re.sub(r"\s+", " ", texto or "").strip()
    if not texto:
        return []
    tokens = texto.split()
    bloco_alto: list[str] = []
    idx = 0
    while idx < len(tokens) and token_assinatura_caixa_alta(tokens[idx]):
        bloco_alto.append(tokens[idx])
        idx += 1
    if len(bloco_alto) >= 2:
        linhas = [" ".join(bloco_alto)]
        restante = " ".join(tokens[idx:]).strip()
        if restante:
            linhas.append(restante)
        return linhas
    return [texto]

def separar_fecho_formal_dos_artigos(blocos: list[str]) -> list[str]:
    resultado: list[str] = []
    for bloco in blocos:
        match = FECHO_FORMAL_RE.search(bloco)
        if not match or match.start() == 0 or not re.match(r"^Art\.", bloco):
            resultado.append(bloco)
            continue
        antes = bloco[: match.start()].rstrip()
        fecho = match.group(0).strip()
        assinaturas = bloco[match.end() :].strip()
        if antes:
            resultado.append(antes)
        resultado.append(fecho)
        resultado.extend(separar_assinaturas(assinaturas))
    return resultado

def finalizar_texto_integral_normativo(urn: str, texto: str, ementa: str | None = None) -> str:
    linhas = linhas_nao_vazias(polir_texto_normativo(texto))
    if not linhas:
        return ""
    titulo_idx = next((idx for idx, linha in enumerate(linhas) if re.match(r"^LEI\b", linha, re.I)), 0)
    titulo = titulo_canonico_por_urn(urn) or linhas[titulo_idx]
    linhas_pos_titulo = remover_nota_publicacao_consolidada(linhas[titulo_idx + 1 :])
    ementa_detectada, idx_pos_ementa = extrair_ementa_topo(linhas_pos_titulo, 0)
    ementa_final = normalizar_ementa(ementa) or ementa_detectada
    preambulo, idx_pos_preambulo = extrair_preambulo(linhas_pos_titulo, idx_pos_ementa)
    corpo_inicio = idx_pos_preambulo if preambulo else idx_pos_ementa
    corpo = separar_fecho_formal_dos_artigos(formatar_blocos_corpo(linhas_pos_titulo[corpo_inicio:]))
    saida = [titulo]
    if ementa_final:
        saida.append(ementa_final)
    if preambulo:
        saida.append(preambulo)
    if corpo:
        saida.append("")
        saida.extend(corpo)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(saida)).strip()

def primeira_linha_util(texto: str) -> str:
    return next((linha.strip() for linha in (texto or "").splitlines() if linha.strip()), "")

def enriquecer_registro_norma_integral(registro: dict[str, Any], urn: str) -> dict[str, Any]:
    info = extrair_info_urn(urn or "")
    json_canonico = registro.get("json_canonico") or {}
    canonical = json_canonico.get("canonical") or {}
    primeira = primeira_linha_util(registro.get("texto_integral") or "")
    registro["tipo_norma"] = "LEI COMPLEMENTAR" if info and info.tipo_norma == "lei.complementar" else "LEI"
    registro["numero"] = canonical.get("numero") or (info.numero if info else None)
    registro["ano"] = canonical.get("ano") or (info.ano if info else None)
    registro["data_assinatura"] = info.data_norma if info else None
    if primeira.upper().startswith("LEI"):
        registro["titulo_norma"] = primeira
    return registro

URN_FIELDS = re.compile(r"urn:lex:br:[^:]+:([^:]+):(\d{4})-(\d{2})-(\d{2});(.+)$")
ANO_RANGES = [
    (2004, 2006, "2004-2006"),
    (2007, 2010, "2007-2010"),
    (2011, 2014, "2011-2014"),
    (2015, 2018, "2015-2018"),
    (2019, 2022, "2019-2022"),
    (2023, 2026, "2023-2026"),
]

@dataclass(frozen=True)
class NormaInfo:
    urn: str
    numero: str
    ano: str
    tipo_norma: str
    data_norma: str

def extrair_info_urn(urn: str) -> NormaInfo | None:
    match = URN_FIELDS.match(urn or "")
    if not match:
        return None
    tipo, ano, mes, dia, numero = match.groups()
    return NormaInfo(urn=urn, numero=numero, ano=ano, tipo_norma=tipo, data_norma=f"{ano}-{mes}-{dia}")

def get_ano_range(ano: int) -> str | None:
    for inicio, fim, label in ANO_RANGES:
        if inicio <= ano <= fim:
            return label
    return None

def numero_com_milhar(numero: str) -> str:
    if len(numero) <= 3:
        return numero
    return f"{numero[:-3]}.{numero[-3:]}"

def candidatos_url_planalto(urn: str) -> list[str]:
    info = extrair_info_urn(urn)
    if not info:
        return []
    try:
        ano_int = int(info.ano)
        num_int = int(info.numero)
    except ValueError:
        return []
    base = "https://www.planalto.gov.br"
    if info.tipo_norma == "lei.complementar":
        raiz = f"{base}/ccivil_03/leis/lcp"
        return [f"{raiz}/Lcp{info.numero}compilado.htm", f"{raiz}/Lcp{info.numero}.htm"]
    if info.tipo_norma == "lei":
        if num_int < 10000:
            raiz = f"{base}/ccivil_03/leis"
            candidatos = [
                f"{raiz}/L{info.numero}compilado.htm",
                f"{raiz}/L{info.numero}consol.htm",
                f"{raiz}/L{info.numero}cons.htm",
                f"{raiz}/L{info.numero}.htm",
            ]
            if 1989 <= ano_int <= 1994:
                raiz_periodo = f"{base}/ccivil_03/leis/1989_1994"
                candidatos.extend(
                    [
                        f"{raiz_periodo}/L{info.numero}compilado.htm",
                        f"{raiz_periodo}/L{info.numero}consol.htm",
                        f"{raiz_periodo}/L{info.numero}cons.htm",
                        f"{raiz_periodo}/L{info.numero}.htm",
                    ]
                )
            return candidatos
        if ano_int == 2001:
            raiz = f"{base}/ccivil_03/leis/leis_2001"
            return [f"{raiz}/l{info.numero}compilado.htm", f"{raiz}/l{info.numero}.htm"]
        if ano_int in {2002, 2003}:
            raiz = f"{base}/ccivil_03/leis/{info.ano}"
            n_milhar = numero_com_milhar(info.numero)
            return [
                f"{raiz}/l{n_milhar}compilado.htm",
                f"{raiz}/l{n_milhar}.htm",
                f"{raiz}/l{info.numero}compilada.htm",
                f"{raiz}/l{info.numero}compilado.htm",
                f"{raiz}/l{info.numero}.htm",
            ]
        if ano_int == 2000:
            raiz = f"{base}/ccivil_03/leis"
            return [
                f"{raiz}/L{info.numero}.htm",
                f"{raiz}/L{info.numero}compilado.htm",
                f"{raiz}/L{info.numero}cons.htm",
            ]
        range_str = get_ano_range(ano_int)
        if range_str:
            raiz = f"{base}/ccivil_03/_ato{range_str}/{info.ano}/lei"
            n_milhar = numero_com_milhar(info.numero)
            candidatos = [f"{raiz}/l{info.numero}compilado.htm", f"{raiz}/l{info.numero}.htm"]
            if n_milhar != info.numero:
                candidatos.extend([f"{raiz}/l{n_milhar}compilado.htm", f"{raiz}/l{n_milhar}.htm"])
            return candidatos
    return []

def inferir_tipo_fonte_normativa(url_origem: str) -> str:
    url = (url_origem or "").casefold()
    if "compilado" in url or "normaatualizada" in url or "consol" in url:
        return "texto_compilado_vigente"
    if "publicacaooriginal" in url:
        return "publicacao_original"
    return "fonte_oficial_sem_classificacao"

def inferir_fonte_preferida(url_origem: str) -> str:
    url = (url_origem or "").casefold()
    if "planalto.gov.br" in url:
        return "planalto"
    if "camara.leg.br" in url or "camara.gov.br" in url:
        return "camara"
    if "senado.leg.br" in url:
        return "senado"
    return "fonte_oficial"

def extrair_titulo_ementa(texto_integral: str, fallback_titulo: str) -> tuple[str, str | None]:
    linhas = [linha.strip() for linha in texto_integral.splitlines() if linha.strip()]
    titulo = next(
        (
            linha
            for linha in linhas[:40]
            if re.match(r"^LEI(\s+COMPLEMENTAR)?\s+N[º°o]?\s*[\d\.]+", linha, re.I)
        ),
        fallback_titulo,
    )
    ementa = None
    if titulo in linhas:
        indice = linhas.index(titulo)
        for offset, linha in enumerate(linhas[indice + 1 : indice + 30], start=indice + 1):
            if re.match(r"^Disp[oõ]e\b", linha, re.I):
                partes = [linha]
                for proxima in linhas[offset + 1 : offset + 8]:
                    if re.match(r"^(PUBLICA|O PRESIDENTE|Fa[cç]o saber|T[IÍ]TULO|CAP[IÍ]TULO|Art\.)", proxima, re.I):
                        break
                    partes.append(proxima)
                ementa = limpar_texto(" ".join(partes))
                break
        if ementa is None:
            for linha in linhas[indice + 1 : indice + 12]:
                if re.match(r"^(Mensagem|Produ[cç][aã]o|Partes mantidas|\(Vide|PUBLICA|O PRESIDENTE|Fa[cç]o saber|T[IÍ]TULO|CAP[IÍ]TULO|Art\.)", linha, re.I):
                    continue
                ementa = linha
                break
    return titulo, ementa

def slugificar_urn(urn: str) -> str:
    slug = unicodedata.normalize("NFKD", (urn or "").lower())
    slug = "".join(c for c in slug if not unicodedata.combining(c))
    slug = re.sub(r"[^a-z0-9]+", "_", slug)
    return re.sub(r"_+", "_", slug).strip("_")

def slugificar_artigo(artigo: str | None) -> str | None:
    if not artigo:
        return None
    artigo = unicodedata.normalize("NFKD", str(artigo).strip())
    artigo = "".join(c for c in artigo if not unicodedata.combining(c))
    artigo = artigo.replace("º", "").replace("°", "").replace("–", "-").lower()
    artigo = re.sub(r"[^0-9a-z\-]+", "_", artigo)
    return re.sub(r"_+", "_", artigo).strip("_").replace("-", "_")

def gerar_chunk_id(urn: str, artigo: str | None, ordem_chunk: int) -> str:
    base = slugificar_urn(urn)
    if artigo:
        return f"{base}__art_{slugificar_artigo(artigo)}__chunk_{ordem_chunk}"
    return f"{base}__chunk_{ordem_chunk}"

def extrair_identificador_artigo(linha: str) -> str | None:
    match = re.match(r"^Art\.\s*([0-9]+(?:[-–][A-Z])?[º°]?)", linha)
    if match:
        return match.group(1).replace("º", "").replace("°", "").replace("–", "-")
    if re.match(r"^Artigo\s+[ÚúUu]nico\b", linha):
        return "unico"
    return None

def montar_texto_contextualizado(titulo_norma: str | None, ementa: str | None, artigo: str | None, texto_chunk: str) -> str:
    partes = []
    if titulo_norma:
        partes.append(titulo_norma)
    if ementa:
        partes.append(ementa)
    if artigo:
        partes.append(f"Art. {artigo}")
    partes.append(texto_chunk)
    return " ".join(partes).strip()

def montar_json_canonico_norma(urn: str, texto_integral: str, url_origem: str, titulo: str | None = None, ementa: str | None = None) -> dict[str, Any]:
    info = extrair_info_urn(urn)
    numero = info.numero if info else ""
    ano = info.ano if info else ""
    fallback_titulo = f"Norma {urn}"
    if titulo is None or ementa is None:
        titulo_detectado, ementa_detectada = extrair_titulo_ementa(texto_integral, fallback_titulo)
        titulo = titulo or titulo_detectado
        ementa = ementa or ementa_detectada
    coletado_em = datetime.now(timezone.utc).isoformat()
    fonte_preferida = inferir_fonte_preferida(url_origem)
    links_oficiais = {fonte_preferida: url_origem, "lexml": f"https://www.lexml.gov.br/urn/{urn}"}
    return {
        "urn": urn,
        "canonical": {"titulo_norma": titulo, "ementa": ementa, "numero": numero, "ano": ano},
        "texto": {
            "fonte_preferida": fonte_preferida,
            "tipo_fonte_normativa": inferir_tipo_fonte_normativa(url_origem),
            "url_origem": url_origem,
            "texto_integral": texto_integral,
            "texto_para_embedding": texto_integral,
        },
        "links_oficiais": links_oficiais,
        "pipeline": {"coletado_em": coletado_em, "versao_pipeline": PIPELINE_VERSION},
    }

def gerar_chunks_artigos(json_canonico: dict[str, Any]) -> list[dict[str, Any]]:
    urn = json_canonico["urn"]
    canonical = json_canonico["canonical"]
    texto_info = json_canonico["texto"]
    pipeline = json_canonico["pipeline"]
    titulo_norma = canonical["titulo_norma"]
    ementa = canonical["ementa"]
    numero = canonical["numero"]
    ano = canonical["ano"]
    fonte = texto_info["fonte_preferida"]
    tipo_fonte_normativa = texto_info.get("tipo_fonte_normativa")
    url_origem = texto_info["url_origem"]
    coletado_em = pipeline["coletado_em"]
    pipeline_version = pipeline["versao_pipeline"]
    linhas = [linha.strip() for linha in texto_info["texto_para_embedding"].splitlines() if linha.strip()]

    chunks: list[dict[str, Any]] = []
    hierarquia_titulo = None
    hierarquia_capitulo = None
    hierarquia_secao = None
    hierarquia_subsecao = None
    artigo_numero = None
    artigo_linhas: list[str] = []
    artigo_hierarquia = {"titulo": None, "capitulo": None, "secao": None, "subsecao": None}
    preambulo: list[str] = []
    ordem_chunk = 0

    def salvar_chunk_artigo() -> None:
        nonlocal ordem_chunk, artigo_numero, artigo_linhas
        if not artigo_numero or not artigo_linhas:
            return
        ordem_chunk += 1
        texto_chunk = limpar_texto("\n".join(artigo_linhas))
        chunks.append(
            {
                "chunk_id": gerar_chunk_id(urn, artigo_numero, ordem_chunk),
                "urn": urn,
                "numero": numero,
                "ano": ano,
                "titulo_norma": titulo_norma,
                "ementa": ementa,
                "fonte": fonte,
                "tipo_bloco": "artigo",
                "artigo": artigo_numero,
                "paragrafo": None,
                "inciso": None,
                "hierarquia_titulo": artigo_hierarquia["titulo"],
                "hierarquia_capitulo": artigo_hierarquia["capitulo"],
                "hierarquia_secao": artigo_hierarquia["secao"],
                "hierarquia_subsecao": artigo_hierarquia["subsecao"],
                "texto_chunk": texto_chunk,
                "texto_contextualizado": montar_texto_contextualizado(titulo_norma, ementa, artigo_numero, texto_chunk),
                "url_origem": url_origem,
                "coletado_em": coletado_em,
                "pipeline_version": pipeline_version,
                "ordem_chunk": ordem_chunk,
                "metadata_json": {},
            }
        )

    for linha in linhas:
        if re.match(r"^T[IÍ]TULO\s+[IVXLCDM]+", linha):
            hierarquia_titulo = linha
            hierarquia_capitulo = None
            hierarquia_secao = None
            hierarquia_subsecao = None
            continue
        if re.match(r"^CAP[IÍ]TULO\s+([IVXLCDM]+|[ÚU]NICO)", linha):
            hierarquia_capitulo = linha
            hierarquia_secao = None
            hierarquia_subsecao = None
            continue
        if re.match(r"^Se[cç][aã]o\s+[IVXLCDM]+", linha, re.I):
            hierarquia_secao = linha
            hierarquia_subsecao = None
            continue
        if re.match(r"^Subse[cç][aã]o\s+[IVXLCDM]+", linha, re.I):
            hierarquia_subsecao = linha
            continue

        artigo_detectado = extrair_identificador_artigo(linha)
        if artigo_detectado:
            if artigo_numero is None and preambulo:
                ordem_chunk += 1
                texto_preambulo = limpar_texto("\n".join(preambulo))
                chunks.append(
                    {
                        "chunk_id": gerar_chunk_id(urn, None, ordem_chunk),
                        "urn": urn,
                        "numero": numero,
                        "ano": ano,
                        "titulo_norma": titulo_norma,
                        "ementa": ementa,
                        "fonte": fonte,
                        "tipo_bloco": "preambulo",
                        "artigo": None,
                        "paragrafo": None,
                        "inciso": None,
                        "hierarquia_titulo": None,
                        "hierarquia_capitulo": None,
                        "hierarquia_secao": None,
                        "hierarquia_subsecao": None,
                        "texto_chunk": texto_preambulo,
                        "texto_contextualizado": montar_texto_contextualizado(titulo_norma, ementa, None, texto_preambulo),
                        "url_origem": url_origem,
                        "coletado_em": coletado_em,
                        "pipeline_version": pipeline_version,
                        "ordem_chunk": ordem_chunk,
                        "metadata_json": {},
                    }
                )
            salvar_chunk_artigo()
            artigo_numero = artigo_detectado
            artigo_linhas = [linha]
            artigo_hierarquia = {
                "titulo": hierarquia_titulo,
                "capitulo": hierarquia_capitulo,
                "secao": hierarquia_secao,
                "subsecao": hierarquia_subsecao,
            }
            continue

        if artigo_numero is None:
            preambulo.append(linha)
        else:
            artigo_linhas.append(linha)

    salvar_chunk_artigo()
    for chunk in chunks:
        metadata = dict(chunk)
        metadata["tipo_fonte_normativa"] = tipo_fonte_normativa
        chunk["metadata_json"] = metadata
    return chunks

def gerar_seed_norma_generica(urn: str, texto_integral: str, url_origem: str, titulo: str | None = None, ementa: str | None = None) -> dict[str, Any]:
    json_canonico = montar_json_canonico_norma(urn, texto_integral, url_origem, titulo, ementa)
    chunks = gerar_chunks_artigos(json_canonico)
    canonical = json_canonico["canonical"]
    texto = json_canonico["texto"]
    pipeline = json_canonico["pipeline"]
    return {
        "registro_norma_integral": {
            "urn": json_canonico["urn"],
            "titulo_norma": canonical["titulo_norma"],
            "ementa": canonical["ementa"],
            "fonte_preferida": texto["fonte_preferida"],
            "url_origem": texto["url_origem"],
            "texto_integral": texto["texto_integral"],
            "quantidade_caracteres": len(texto["texto_integral"]),
            "coletado_em": pipeline["coletado_em"],
            "pipeline_version": pipeline["versao_pipeline"],
            "json_canonico": json_canonico,
        },
        "chunks_norma": chunks,
    }

def candidatos_url_camara_via_lexml(urn: str) -> list[str]:
    try:
        html_lexml, _ = request_text(f"https://www.lexml.gov.br/urn/{urn}")
    except Exception:
        return []

    urls: list[str] = []
    for raw_url in re.findall(r"https?://[^\"'\s<>]+", html_lexml):
        url = html.unescape(raw_url).rstrip("](),.;")
        if "camara" not in url or "/legin/fed/lei/" not in url:
            continue
        if url not in urls:
            urls.append(url)

    def prioridade(url: str) -> tuple[int, str]:
        if "publicacaooriginal" in url:
            return (0, url)
        if "norma" in url:
            return (1, url)
        return (2, url)

    return sorted(urls, key=prioridade)

def coletar_inteiro_teor_planalto(urn: str) -> dict[str, Any]:
    erros: list[dict[str, str]] = []
    candidatos = candidatos_url_planalto(urn)
    if not candidatos:
        raise RuntimeError(f"Nenhum candidato de URL do Planalto para {urn}")
    for url in candidatos:
        try:
            html_text, url_final = request_text(url)
            texto = polir_texto_normativo(html_para_texto_normativo_generico(html_text))
            if len(texto) < 300:
                raise RuntimeError("Texto extraido ficou pequeno demais.")
            if PAUSA_PLANALTO_SEGUNDOS:
                time.sleep(PAUSA_PLANALTO_SEGUNDOS)
            return {"url_origem": url_final, "texto_integral": texto, "hash_texto": sha256_text(texto), "erros_fontes": erros}
        except Exception as exc:
            erros.append({"url": url, "erro": str(exc)})
    for url in candidatos_url_camara_via_lexml(urn):
        try:
            html_text, url_final = request_text(url)
            texto = polir_texto_normativo(html_para_texto_normativo_generico(html_text))
            if len(texto) < 300:
                raise RuntimeError("Texto extraido ficou pequeno demais.")
            if PAUSA_PLANALTO_SEGUNDOS:
                time.sleep(PAUSA_PLANALTO_SEGUNDOS)
            return {"url_origem": url_final, "texto_integral": texto, "hash_texto": sha256_text(texto), "erros_fontes": erros}
        except Exception as exc:
            erros.append({"url": url, "erro": str(exc)})
    raise RuntimeError(f"Nenhuma fonte oficial funcionou para {urn}: {erros}")

def gerar_seed_processada(item: dict[str, Any]) -> dict[str, Any]:
    urn = item.get("urn")
    if not urn:
        raise ValueError("Item sem URN nao pode ser coletado com seguranca.")
    coleta = coletar_inteiro_teor_planalto(urn)
    texto_integral = finalizar_texto_integral_normativo(urn, coleta["texto_integral"], item.get("ementa"))
    hash_texto = sha256_text(texto_integral)
    seed = gerar_seed_norma_generica(
        urn=urn,
        texto_integral=texto_integral,
        url_origem=coleta["url_origem"],
        titulo=primeira_linha_util(texto_integral) or item.get("norma_nome"),
        ementa=item.get("ementa"),
    )
    norma = seed["registro_norma_integral"]
    enriquecer_registro_norma_integral(norma, urn)
    norma["hash_texto"] = hash_texto
    norma["metadados_senado"] = {key: value for key, value in item.items() if key != "raw_senado"}
    norma["raw_senado"] = item.get("raw_senado")
    norma["erros_fontes"] = coleta["erros_fontes"]
    for chunk in seed["chunks_norma"]:
        chunk["hash_texto_norma"] = hash_texto
    return seed

def item_key(item: dict[str, Any]) -> str:
    return f"{item.get('tipo_norma', 'LEI')}:{item.get('numero')}:{item.get('ano')}"

def load_checkpoint() -> dict[str, Any]:
    if PATHS["checkpoint"].exists():
        return json.loads(PATHS["checkpoint"].read_text(encoding="utf-8"))
    return {"pipeline_version": PIPELINE_VERSION, "items": {}, "updated_at": None}

def save_checkpoint(checkpoint: dict[str, Any]) -> None:
    checkpoint["updated_at"] = datetime.now(timezone.utc).isoformat()
    PATHS["checkpoint"].write_text(json.dumps(checkpoint, ensure_ascii=False, indent=2), encoding="utf-8")

def update_checkpoint(checkpoint: dict[str, Any], item: dict[str, Any], status: str, **extra: Any) -> dict[str, Any]:
    key = item_key(item)
    atual = checkpoint.setdefault("items", {}).get(key, {})
    atual.update(
        {
            "status": status,
            "tipo_norma": item.get("tipo_norma", "LEI"),
            "numero": item.get("numero"),
            "ano": item.get("ano"),
            "urn": extra.pop("urn", item.get("urn")),
            "updated_at": datetime.now(timezone.utc).isoformat(),
            **extra,
        }
    )
    checkpoint["items"][key] = atual
    save_checkpoint(checkpoint)
    return checkpoint

def get_item_status(checkpoint: dict[str, Any], item: dict[str, Any]) -> str | None:
    return checkpoint.get("items", {}).get(item_key(item), {}).get("status")

def gerar_manifest(force: bool = False) -> dict[str, Any]:
    if PATHS["manifest"].exists() and not force:
        leis = read_jsonl(PATHS["manifest"])
    else:
        leis = descobrir_leis_intervalo()
        write_jsonl(PATHS["manifest"], leis)
    resumo_por_ano: dict[str, int] = {}
    for item in leis:
        ano = str(item.get("ano"))
        resumo_por_ano[ano] = resumo_por_ano.get(ano, 0) + 1
    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "ano_inicio": ANO_INICIO,
        "ano_fim": ANO_FIM,
        "total_leis": len(leis),
        "resumo_por_ano": dict(sorted(resumo_por_ano.items(), reverse=True)),
        "manifest_path": str(PATHS["manifest"]),
    }
    PATHS["manifest_summary"].write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary

def criar_lotes_manifest(tamanho_lote: int = TAMANHO_LOTE_MANIFEST) -> dict[str, Any]:
    manifest = read_jsonl(PATHS["manifest"])
    if not manifest:
        raise RuntimeError("Manifest vazio. Execute gerar_manifest primeiro.")
    BATCHES_DIR.mkdir(parents=True, exist_ok=True)
    for old_file in BATCHES_DIR.glob("manifest_lote_*.jsonl"):
        old_file.unlink()
    arquivos: list[str] = []
    for indice, inicio in enumerate(range(0, len(manifest), tamanho_lote), start=1):
        path = BATCHES_DIR / f"manifest_lote_{indice:04d}.jsonl"
        write_jsonl(path, manifest[inicio : inicio + tamanho_lote])
        arquivos.append(str(path))
    summary = {"tamanho_lote": tamanho_lote, "total_leis": len(manifest), "total_lotes": len(arquivos), "primeiro_lote": arquivos[0], "ultimo_lote": arquivos[-1]}
    (PROCESSED_DIR / "batches_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary

def processar_manifest(limite: int | None = LIMITE_COLETA_AMOSTRA, pular_salvos: bool = True) -> dict[str, Any]:
    manifest = read_jsonl(PATHS["manifest"])
    if not manifest:
        raise RuntimeError("Manifest vazio. Execute gerar_manifest primeiro.")
    checkpoint = load_checkpoint()
    processadas = 0
    falhas = 0
    puladas = 0
    tentadas = 0
    for item in manifest:
        if limite is not None and tentadas >= limite:
            break
        if pular_salvos and get_item_status(checkpoint, item) == "saved":
            puladas += 1
            continue
        tentadas += 1
        try:
            checkpoint = update_checkpoint(checkpoint, item, "metadata_loading")
            try:
                detalhe = consultar_detalhe_lei(numero=str(item["numero"]), ano=str(item["ano"]), tipo=item.get("tipo_norma", "LEI"))
                item_enriquecido = {**item, **detalhe, "status": "metadata_loaded"}
            except Exception as detalhe_exc:
                if not item.get("urn"):
                    raise
                item_enriquecido = {**item, "status": "metadata_loaded", "metadata_fallback": True, "metadata_fallback_error": str(detalhe_exc), "raw_senado": None}
            checkpoint = update_checkpoint(
                checkpoint,
                item_enriquecido,
                "metadata_loaded",
                urn=item_enriquecido.get("urn"),
                metadata_fallback=item_enriquecido.get("metadata_fallback", False),
                metadata_fallback_error=item_enriquecido.get("metadata_fallback_error"),
            )
            checkpoint = update_checkpoint(checkpoint, item_enriquecido, "collecting_text")
            seed = gerar_seed_processada(item_enriquecido)
            norma = seed["registro_norma_integral"]
            chunks = seed["chunks_norma"]
            append_jsonl(PATHS["normas"], norma)
            for chunk in chunks:
                append_jsonl(PATHS["chunks"], chunk)
            checkpoint = update_checkpoint(checkpoint, item_enriquecido, "saved", urn=item_enriquecido.get("urn"), chunks=len(chunks))
            processadas += 1
            print(f"OK {processadas}: Lei {item_enriquecido.get('numero')}/{item_enriquecido.get('ano')} - chunks {len(chunks)}")
        except Exception as exc:
            falhas += 1
            append_jsonl(PATHS["falhas"], {"tipo_norma": item.get("tipo_norma", "LEI"), "numero": item.get("numero"), "ano": item.get("ano"), "urn": item.get("urn"), "erro": str(exc)})
            checkpoint = update_checkpoint(checkpoint, item, "failed", ultimo_erro=str(exc))
            print(f"FALHA Lei {item.get('numero')}/{item.get('ano')}: {exc}")
    return {"limite": limite, "tentadas": tentadas, "processadas": processadas, "falhas": falhas, "puladas": puladas, "normas_path": str(PATHS["normas"]), "chunks_path": str(PATHS["chunks"]), "checkpoint_path": str(PATHS["checkpoint"])}

def validar_amostra_processada() -> dict[str, Any]:
    normas = read_jsonl(PATHS["normas"])
    chunks = read_jsonl(PATHS["chunks"])
    problems: list[dict[str, Any]] = []
    required_norma = {"urn", "tipo_norma", "numero", "ano", "data_assinatura", "titulo_norma", "ementa", "fonte_preferida", "url_origem", "texto_integral", "quantidade_caracteres", "hash_texto", "coletado_em", "pipeline_version", "json_canonico"}
    required_chunk = {"chunk_id", "urn", "numero", "ano", "titulo_norma", "texto_chunk", "texto_contextualizado", "url_origem", "ordem_chunk"}

    def add_problem(category: str, message: str, ref: str | None = None) -> None:
        problems.append({"category": category, "message": message, "ref": ref})

    titulo_incompleto_re = re.compile(r"^LEI(?:\s+N[º°o]?)?$", re.I)
    titulo_data_quebrada_re = re.compile(r"^LEI(?:\s+COMPLEMENTAR)?\s+N[º°o]?\s*[0-9][^\n]*,\s*DE$", re.I)
    artigo_sem_caput_re = re.compile(r"(?m)^Art\.\s*[0-9]+(?:[-–][A-Z])?[º°]?\s*$")
    faco_saber_quebrado_re = re.compile(r"(?im)^Fa[çc]o\s*$\n^\s*saber\b")
    seguinte_lei_quebrado_re = re.compile(r"(?im)^Fa[çc]o saber[^\n]*\ba\s*$\n^\s*seguinte Lei:")
    ruido_topo_re = re.compile(
        r"^(Mensagem(?:\s+de\s+veto)?|Produ[çc][aã]o\s+de\s+efeito|"
        r"Partes\s+mantidas|Vig[êé]ncia|Regulamento|Texto\s+compilado|"
        r"Convers[ãa]o\s+da\s+Medida|Convers[ãa]o\s+da\s+MP|"
        r"\(Vide\b.*\)|PUBLICA[ÇC][ÃA]O\s+CONSOLIDADA)",
        re.I,
    )
    fonte_dominios = {
        "planalto": ("planalto.gov.br",),
        "camara": ("camara.leg.br", "camara.gov.br"),
        "senado": ("senado.leg.br",),
    }

    def fonte_preferida_esperada(url_origem: str) -> str:
        url = (url_origem or "").casefold()
        for fonte, dominios in fonte_dominios.items():
            if any(dominio in url for dominio in dominios):
                return fonte
        return "desconhecida"

    urns = [norma.get("urn") for norma in normas]
    chunk_ids = [chunk.get("chunk_id") for chunk in chunks]
    for urn, count in Counter(urns).items():
        if urn and count > 1:
            add_problem("norma_duplicada", f"URN duplicada {count} vezes.", urn)
    for chunk_id, count in Counter(chunk_ids).items():
        if chunk_id and count > 1:
            add_problem("chunk_duplicado", f"chunk_id duplicado {count} vezes.", chunk_id)

    chunks_by_urn: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for chunk in chunks:
        if chunk.get("urn"):
            chunks_by_urn[chunk["urn"]].append(chunk)

    for norma in normas:
        urn = norma.get("urn")
        missing = required_norma - set(norma)
        if missing:
            add_problem("norma_schema", f"Campos ausentes: {sorted(missing)}", urn)
        texto = norma.get("texto_integral") or ""
        linhas = [linha.strip() for linha in texto.splitlines() if linha.strip()]
        primeira_linha = linhas[0] if linhas else ""
        if len(texto) < 300:
            add_problem("texto_curto", "Texto integral tem menos de 300 caracteres.", urn)
        if titulo_incompleto_re.fullmatch(primeira_linha):
            add_problem("titulo_quebrado", "Primeira linha ficou com titulo normativo incompleto.", urn)
        if titulo_data_quebrada_re.fullmatch(primeira_linha):
            add_problem("titulo_quebrado", "Data do titulo normativo ficou separada da primeira linha.", urn)
        if "\n\n\n" in texto:
            add_problem("quebras_excessivas", "Texto tem tres ou mais quebras seguidas.", urn)
        url_origem = norma.get("url_origem") or ""
        fonte_esperada = fonte_preferida_esperada(url_origem)
        if fonte_esperada == "desconhecida":
            add_problem("fonte", "URL de origem nao aponta para fonte oficial reconhecida.", urn)
        elif norma.get("fonte_preferida") != fonte_esperada:
            add_problem("fonte", f"fonte_preferida deveria ser {fonte_esperada!r} para a URL de origem.", urn)
        if "Art.\n" in texto or "Art.\r" in texto:
            add_problem("limpeza_artigo", "Marcador Art. ainda esta quebrado por linha.", urn)
        if artigo_sem_caput_re.search(texto):
            add_problem("limpeza_artigo", "Marcador de artigo ficou sozinho em uma linha.", urn)
        if faco_saber_quebrado_re.search(texto):
            add_problem("preambulo_quebrado", "Preambulo ficou com 'Faco' separado de 'saber'.", urn)
        if seguinte_lei_quebrado_re.search(texto):
            add_problem("preambulo_quebrado", "Preambulo ficou com 'a' separado de 'seguinte Lei'.", urn)
        if any(ruido_topo_re.match(linha) for linha in linhas[1:20]):
            add_problem("limpeza_ruido_topo", "Texto integral ainda contem ruido de topo do portal oficial.", urn)
        if any(re.fullmatch(r"Disp[oõ]e", linha, re.I) for linha in linhas[1:20]):
            add_problem("ementa_fragmentada", "Ementa ficou fragmentada com linha isolada do tipo 'Dispoe'.", urn)
        if norma.get("hash_texto") and norma.get("hash_texto") != sha256_text(texto):
            add_problem("hash", "Hash do texto integral nao confere.", urn)
        if int(norma.get("quantidade_caracteres") or 0) != len(texto):
            add_problem("tamanho", "Quantidade de caracteres diverge do texto.", urn)
        norma_chunks = chunks_by_urn.get(urn or "", [])
        if not norma_chunks:
            add_problem("chunk_ausente", "Norma sem chunks.", urn)
        artigo_markers = re.findall(r"(?m)^Art\.\s*[0-9]+", texto)
        artigo_chunks = [chunk for chunk in norma_chunks if chunk.get("tipo_bloco") == "artigo"]
        if artigo_markers and not artigo_chunks:
            add_problem("chunk_artigo", "Texto tem artigos, mas nenhum chunk de artigo.", urn)

    norma_urns = set(urn for urn in urns if urn)
    for chunk in chunks:
        ref = chunk.get("chunk_id")
        missing = required_chunk - set(chunk)
        if missing:
            add_problem("chunk_schema", f"Campos ausentes: {sorted(missing)}", ref)
        if chunk.get("urn") not in norma_urns:
            add_problem("chunk_orfao", "Chunk referencia URN sem norma integral.", ref)
        if not (chunk.get("texto_chunk") or "").strip():
            add_problem("chunk_texto", "Chunk sem texto_chunk.", ref)
        if not (chunk.get("texto_contextualizado") or "").strip():
            add_problem("chunk_texto", "Chunk sem texto_contextualizado.", ref)
        texto_chunk = chunk.get("texto_chunk") or ""
        primeira_linha_chunk = next((linha.strip() for linha in texto_chunk.splitlines() if linha.strip()), "")
        if titulo_incompleto_re.fullmatch(primeira_linha_chunk):
            add_problem("chunk_titulo_quebrado", "Chunk inicia com titulo normativo incompleto.", ref)
        if titulo_data_quebrada_re.fullmatch(primeira_linha_chunk):
            add_problem("chunk_titulo_quebrado", "Chunk inicia com data do titulo normativo separada.", ref)
        if artigo_sem_caput_re.search(texto_chunk):
            add_problem("chunk_artigo_quebrado", "Chunk possui marcador de artigo sozinho em uma linha.", ref)

    summary = {
        "normas": len(normas),
        "chunks": len(chunks),
        "urns_unicas": len(set(urns)),
        "chunk_ids_unicos": len(set(chunk_ids)),
        "fontes_por_origem": dict(Counter(fonte_preferida_esperada(norma.get("url_origem") or "") for norma in normas)),
        "problemas": len(problems),
        "problemas_por_categoria": dict(Counter(p["category"] for p in problems)),
        "aprovado": len(problems) == 0,
    }
    return {"summary": summary, "problems": problems}

def salvar_relatorio_validacao() -> dict[str, Any]:
    result = validar_amostra_processada()
    PATHS["validacao_json"].write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    summary = result["summary"]
    linhas = [
        "# Relatorio de Validacao da Amostra",
        "",
        f"- Normas integrais: {summary['normas']}",
        f"- Chunks juridicos: {summary['chunks']}",
        f"- URNs unicas: {summary['urns_unicas']}",
        f"- Chunk IDs unicos: {summary['chunk_ids_unicos']}",
        f"- Problemas encontrados: {summary['problemas']}",
        f"- Aprovado: {summary['aprovado']}",
        "",
        "## Fontes por origem",
        "",
    ]
    if summary["fontes_por_origem"]:
        for fonte, count in summary["fontes_por_origem"].items():
            linhas.append(f"- {fonte}: {count}")
    else:
        linhas.append("- Nenhuma fonte encontrada.")
    linhas.extend([
        "",
        "## Problemas por categoria",
        "",
    ])
    if summary["problemas_por_categoria"]:
        for category, count in summary["problemas_por_categoria"].items():
            linhas.append(f"- {category}: {count}")
    else:
        linhas.append("- Nenhum problema encontrado.")
    if result["problems"]:
        linhas.extend(["", "## Primeiros problemas", ""])
        for problem in result["problems"][:30]:
            linhas.append(f"- `{problem['category']}`: {problem['message']} ({problem.get('ref') or 'sem referencia'})")
    PATHS["validacao_md"].write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return {"result": result, "json_path": str(PATHS["validacao_json"]), "md_path": str(PATHS["validacao_md"])}

def metrics_norma(norma: dict[str, Any], chunks: list[dict[str, Any]]) -> dict[str, Any]:
    texto = norma.get("texto_integral") or ""
    linhas = [linha.strip() for linha in texto.splitlines() if linha.strip()]
    ruido_topo_re = re.compile(
        r"^(Mensagem(?:\s+de\s+veto)?|Produ[çc][aã]o\s+de\s+efeito|"
        r"Partes\s+mantidas|Vig[êé]ncia|Regulamento|Texto\s+compilado|"
        r"Convers[ãa]o\s+da\s+Medida|Convers[ãa]o\s+da\s+MP|"
        r"\(Vide\b.*\)|PUBLICA[ÇC][ÃA]O\s+CONSOLIDADA)",
        re.I,
    )
    return {
        "urn": norma.get("urn"),
        "titulo": norma.get("titulo_norma"),
        "primeira_linha": linhas[0] if linhas else None,
        "caracteres": len(texto),
        "linhas_nao_vazias": len(linhas),
        "marcadores_artigo": len(re.findall(r"(?m)^Art\.\s*[0-9]+", texto)),
        "chunks": len(chunks),
        "chunks_artigo": sum(1 for chunk in chunks if chunk.get("tipo_bloco") == "artigo"),
        "tem_art_quebrado": "Art.\n" in texto or "Art.\r" in texto,
        "tem_titulo_quebrado": (linhas[0].upper() == "LEI") if linhas else True,
        "tem_ordinal_nao_normalizado": bool(re.search(r"\b[Nn]\s+o\b", texto)),
        "tem_substituicao_unicode": "\ufffd" in texto,
        "tem_ruido_topo": any(ruido_topo_re.match(linha) for linha in linhas[1:20]),
        "tem_ementa_fragmentada": any(re.fullmatch(r"Disp[oõ]e", linha, re.I) for linha in linhas[1:20]),
    }

def aprovado_padrao_fino(metrics: dict[str, Any]) -> bool:
    return (
        not metrics["tem_art_quebrado"]
        and not metrics["tem_titulo_quebrado"]
        and not metrics["tem_ordinal_nao_normalizado"]
        and not metrics["tem_substituicao_unicode"]
        and not metrics["tem_ruido_topo"]
        and not metrics["tem_ementa_fragmentada"]
        and metrics["chunks"] > 0
    )




def auditar_padrao_fino_processado() -> dict[str, Any]:
    normas = read_jsonl(PATHS["normas"])
    chunks = read_jsonl(PATHS["chunks"])
    chunks_by_urn: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for chunk in chunks:
        chunks_by_urn[chunk.get("urn")].append(chunk)

    issues: list[dict[str, Any]] = []
    amostras: list[dict[str, Any]] = []
    indices = sorted({0, len(normas) - 1, len(normas) // 2}) if normas else []
    urns_amostra = [normas[idx].get("urn") for idx in indices if 0 <= idx < len(normas)]

    for norma in normas:
        urn = norma.get("urn")
        metrics = metrics_norma(norma, chunks_by_urn.get(urn, []))
        if not aprovado_padrao_fino(metrics):
            issues.append({"urn": urn, "metrics": metrics})
        if urn in urns_amostra:
            amostras.append({**metrics, "aprovado_padrao_fino": aprovado_padrao_fino(metrics)})

    result = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "summary": {
            "normas": len(normas),
            "chunks": len(chunks),
            "issues": len(issues),
            "amostras": len(amostras),
            "aprovado": len(issues) == 0,
        },
        "issues": issues[:100],
        "amostras": amostras,
    }
    PATHS["auditoria_fina_json"].write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    linhas = [
        "# Auditoria de Padrao Fino com Base na Lei 8.112",
        "",
        f"- Normas avaliadas: {result['summary']['normas']}",
        f"- Chunks avaliados: {result['summary']['chunks']}",
        f"- Issues: {result['summary']['issues']}",
        f"- Aprovado: {result['summary']['aprovado']}",
        "",
        "## Amostras",
        "",
    ]
    for item in amostras:
        linhas.append(
            f"- `{item['urn']}` | chunks={item['chunks']} | "
            f"art_quebrado={item['tem_art_quebrado']} | "
            f"ruido_topo={item['tem_ruido_topo']} | "
            f"ementa_fragmentada={item['tem_ementa_fragmentada']} | "
            f"aprovado={item['aprovado_padrao_fino']}"
        )
    if issues:
        linhas.extend(["", "## Primeiras issues", ""])
        for issue in issues[:30]:
            linhas.append(f"- `{issue['urn']}`: {issue['metrics']}")
    else:
        linhas.extend(["", "## Issues", "", "- Nenhuma issue encontrada."])
    PATHS["auditoria_fina_md"].write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return {"result": result, "json_path": str(PATHS["auditoria_fina_json"]), "md_path": str(PATHS["auditoria_fina_md"])}

def dividir_jsonl_em_shards(input_path: Path, output_dir: Path, prefixo: str, tamanho_shard: int) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    for old_file in output_dir.glob(f"{prefixo}_*.jsonl"):
        old_file.unlink()
    arquivos: list[str] = []
    shard_index = 1
    count_no_shard = 0
    file = None
    try:
        with input_path.open("r", encoding="utf-8") as source:
            for line in source:
                if not line.strip():
                    continue
                if file is None or count_no_shard >= tamanho_shard:
                    if file is not None:
                        file.close()
                    path = output_dir / f"{prefixo}_{shard_index:04d}.jsonl"
                    file = path.open("w", encoding="utf-8")
                    arquivos.append(str(path))
                    shard_index += 1
                    count_no_shard = 0
                file.write(line)
                count_no_shard += 1
    finally:
        if file is not None:
            file.close()
    return arquivos

def gerar_shards_jsonl() -> dict[str, Any]:
    normas_shards = dividir_jsonl_em_shards(PATHS["normas"], SHARDS_DIR / "normas", "normas", TAMANHO_SHARD_NORMAS)
    chunks_shards = dividir_jsonl_em_shards(PATHS["chunks"], SHARDS_DIR / "chunks", "chunks", TAMANHO_SHARD_CHUNKS)
    summary = {
        "normas_shards": normas_shards,
        "chunks_shards": chunks_shards,
        "total_normas_shards": len(normas_shards),
        "total_chunks_shards": len(chunks_shards),
        "formato": "jsonl",
        "pipeline_version": PIPELINE_VERSION,
    }
    (SHARDS_DIR / "shards_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary

print("Biblioteca interna carregada.")


Biblioteca interna carregada.


### Checagem da biblioteca

Executa uma normalizacao controlada e confirma que a URN de referencia resolve candidatos oficiais de texto integral. A checagem cobre os dois pontos mais sensiveis da biblioteca antes da coleta: acabamento textual e montagem de rota oficial.


In [ ]:
texto_teste = """
LEI
Nº 15.407, DE 11 DE MAIO DE 2026

Art.
1
º Esta Lei entra em vigor.
"""
texto_limpo = polir_texto_normativo(texto_teste)
assert texto_limpo.startswith("LEI Nº 15.407")
assert "Art. 1º" in texto_limpo
assert candidatos_url_planalto(URN_REFERENCIA_VALIDACAO)
print(texto_limpo)
print("Biblioteca aprovada.")


LEI Nº 15.407, DE 11 DE MAIO DE 2026

Art. 1º Esta Lei entra em vigor.
Biblioteca aprovada.


## 3. Manifest de leis

O manifest nasce da descoberta na API oficial em ordem decrescente, de 2026 ate 1988. No modo amostra, o notebook seleciona uma quantidade menor desse mesmo manifest para reduzir tempo de execucao; no modo completo, todos os itens descobertos seguem para coleta.

O arquivo `manifest_leis.jsonl` e a fronteira entre descoberta e captura. Cada linha deve conter numero, ano, data, URN e metadados suficientes para consultar detalhes e resolver o inteiro teor. A contagem total descoberta fica registrada no resumo para diferenciar volume disponivel de volume selecionado.


In [ ]:
def limpar_saidas_processadas() -> None:
    if not RESETAR_ARQUIVOS_PROCESSADOS:
        return

    # Reset controlado: remove apenas artefatos derivados desta execucao.
    # A biblioteca e a configuracao permanecem intactas.
    arquivos = [
        PATHS["manifest"],
        PATHS["manifest_summary"],
        PATHS["checkpoint"],
        PATHS["normas"],
        PATHS["chunks"],
        PATHS["falhas"],
        PATHS["validacao_json"],
        PATHS["validacao_md"],
        PATHS["cobertura_json"],
        PATHS["auditoria_fina_json"],
        PATHS["auditoria_fina_md"],
        PATHS["embeddings"],
        PATHS["embedding_summary"],
        PATHS["pgvector_summary"],
    ]
    for path in arquivos:
        if path.exists():
            path.unlink()

    for directory, pattern in [
        (BATCHES_DIR, "manifest_lote_*.jsonl"),
        (REPORTS_DIR, "texto_integral_validacao_*.txt"),
    ]:
        for path in directory.glob(pattern):
            path.unlink()


def manifest_item_from_urn(urn: str) -> dict[str, Any]:
    info = extrair_info_urn(urn)
    if info is None:
        raise ValueError(f"URN invalida para amostra: {urn}")
    return {
        "tipo_norma": "LEI",
        "numero": info.numero,
        "ano": info.ano,
        "data_assinatura": info.data_norma,
        "norma": None,
        "norma_nome": None,
        "ementa": None,
        "urn": urn,
        "senado_id": "",
        "status": "discovered",
        "tentativas": 0,
        "ultimo_erro": None,
    }


def inteiro_ordem_manifest(value: Any) -> int:
    digits = re.sub(r"\D", "", str(value or ""))
    return int(digits or 0)


def chave_ordem_manifest(item: dict[str, Any]) -> tuple[int, str, int, str]:
    return (
        inteiro_ordem_manifest(item.get("ano")),
        item.get("data_assinatura") or "",
        inteiro_ordem_manifest(item.get("numero")),
        item.get("urn") or "",
    )


def ordenar_manifest(manifest: list[dict[str, Any]]) -> list[dict[str, Any]]:
    # Mantem o contrato herdado do pipeline unificado: manifest salvo em ordem decrescente.
    return sorted(manifest, key=chave_ordem_manifest, reverse=True)


def selecionar_manifest_amostra(manifest_completo: list[dict[str, Any]]) -> list[dict[str, Any]]:
    por_urn = {item.get("urn"): item for item in manifest_completo if item.get("urn")}
    if SAMPLE_URNS:
        # Quando a amostra e manual, todas as URNs precisam existir na descoberta oficial.
        faltantes = [urn for urn in SAMPLE_URNS if urn not in por_urn]
        if faltantes:
            raise RuntimeError(f"URNs da amostra nao encontradas no manifest descoberto: {faltantes}")
        limite_manual = SAMPLE_LIMIT if SAMPLE_LIMIT > 0 else len(SAMPLE_URNS)
        return ordenar_manifest([por_urn[urn] for urn in SAMPLE_URNS[:limite_manual]])

    if SAMPLE_LIMIT <= 0:
        raise ValueError("SAMPLE_LIMIT precisa ser maior que zero no modo amostra.")

    selecionadas: list[dict[str, Any]] = []
    urns_usadas: set[str] = set()
    referencia = por_urn.get(URN_REFERENCIA_VALIDACAO) if URN_REFERENCIA_VALIDACAO else None
    if referencia:
        # Mantem uma norma previsivel para inspecao de inteiro teor na amostra.
        selecionadas.append(referencia)
        urns_usadas.add(URN_REFERENCIA_VALIDACAO)

    # Completa a amostra com a ordem natural da API: leis mais recentes primeiro.
    for item in manifest_completo:
        urn = item.get("urn")
        if not urn or urn in urns_usadas:
            continue
        selecionadas.append(item)
        urns_usadas.add(urn)
        if len(selecionadas) >= SAMPLE_LIMIT:
            break

    return ordenar_manifest(selecionadas)


limpar_saidas_processadas()

# A descoberta completa sempre acontece antes do corte da amostra.
# Assim SAMPLE_LIMIT altera apenas o volume processado, nao a fonte do manifest.
manifest_completo = ordenar_manifest(descobrir_leis_intervalo(ANO_INICIO, ANO_FIM))

if RODAR_CORPUS_COMPLETO:
    manifest = manifest_completo
else:
    manifest = selecionar_manifest_amostra(manifest_completo)

manifest = ordenar_manifest(manifest)
write_jsonl(PATHS["manifest"], manifest)

resumo_por_ano: dict[str, int] = {}
for item in manifest:
    ano = str(item.get("ano"))
    resumo_por_ano[ano] = resumo_por_ano.get(ano, 0) + 1

manifest_summary = {
    "pipeline_version": PIPELINE_VERSION,
    "modo": NOTEBOOK_MODE,
    "rodar_corpus_completo": RODAR_CORPUS_COMPLETO,
    "ano_inicio": ANO_INICIO,
    "ano_fim": ANO_FIM,
    "total_leis_descobertas": len(manifest_completo),
    "total_leis": len(manifest),
    "sample_urns": SAMPLE_URNS,
    "sample_limit": SAMPLE_LIMIT,
    "urn_referencia_validacao": URN_REFERENCIA_VALIDACAO,
    "resumo_por_ano": dict(sorted(resumo_por_ano.items(), reverse=True)),
    "manifest_path": str(PATHS["manifest"]),
}
PATHS["manifest_summary"].write_text(
    json.dumps(manifest_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest_summary, ensure_ascii=False, indent=2))


Descobrindo leis de 2026...
Descobrindo leis de 2025...
Descobrindo leis de 2024...
Descobrindo leis de 2023...
Descobrindo leis de 2022...
Descobrindo leis de 2021...
Descobrindo leis de 2020...
Descobrindo leis de 2019...
Descobrindo leis de 2018...
Descobrindo leis de 2017...
Descobrindo leis de 2016...
Descobrindo leis de 2015...
Descobrindo leis de 2014...
Descobrindo leis de 2013...
Descobrindo leis de 2012...
Descobrindo leis de 2011...
Descobrindo leis de 2010...
Descobrindo leis de 2009...
Descobrindo leis de 2008...
Descobrindo leis de 2007...
Descobrindo leis de 2006...
Descobrindo leis de 2005...
Descobrindo leis de 2004...
Descobrindo leis de 2003...
Descobrindo leis de 2002...
Descobrindo leis de 2001...
Descobrindo leis de 2000...
Descobrindo leis de 1999...
Descobrindo leis de 1998...
Descobrindo leis de 1997...
Descobrindo leis de 1996...
Descobrindo leis de 1995...
Descobrindo leis de 1994...
Descobrindo leis de 1993...
Descobrindo leis de 1992...
Descobrindo leis de 

### Checagem do manifest

Confirma contagem, ordenacao, campos obrigatorios e selecao da amostra. No completo, a checagem tambem exige volume compativel com o periodo informado, evitando seguir com resposta parcial da API.


In [ ]:
manifest = read_jsonl(PATHS["manifest"])
assert len(manifest) == manifest_summary["total_leis"]
assert all(item.get("numero") for item in manifest)
assert all(item.get("ano") for item in manifest)

pares_ordem = [chave_ordem_manifest(item) for item in manifest]
assert pares_ordem == sorted(pares_ordem, reverse=True)

if RODAR_CORPUS_COMPLETO:
    assert len(manifest) == manifest_summary["total_leis_descobertas"]
    assert len(manifest) > 1000, "Manifest completo ficou pequeno demais para o periodo informado."
    print("Manifest completo aprovado.")
    print("Total de leis:", len(manifest))
    print("Primeiras 10 leis:")
    itens_preview = manifest[:10]
else:
    assert manifest_summary["total_leis_descobertas"] > len(manifest)
    esperado = min(SAMPLE_LIMIT, len(SAMPLE_URNS)) if SAMPLE_URNS else min(SAMPLE_LIMIT, manifest_summary["total_leis_descobertas"])
    assert len(manifest) == esperado
    if SAMPLE_URNS:
        assert {item["urn"] for item in manifest} == set(SAMPLE_URNS[:esperado])
    elif URN_REFERENCIA_VALIDACAO:
        assert any(item.get("urn") == URN_REFERENCIA_VALIDACAO for item in manifest)
    print("Manifest da amostra aprovado.")
    print("Total descoberto no periodo:", manifest_summary["total_leis_descobertas"])
    print("Total selecionado para amostra:", len(manifest))
    itens_preview = manifest

for item in itens_preview:
    print(item.get("numero"), item.get("ano"), item.get("data_assinatura"), item.get("urn"))


Manifest completo aprovado.
Total de leis: 7770
Primeiras 10 leis:
15417 2026 2026-05-25 urn:lex:br:federal:lei:2026-05-25;15417
15416 2026 2026-05-25 urn:lex:br:federal:lei:2026-05-25;15416
15415 2026 2026-05-25 urn:lex:br:federal:lei:2026-05-25;15415
15414 2026 2026-05-21 urn:lex:br:federal:lei:2026-05-21;15414
15413 2026 2026-05-21 urn:lex:br:federal:lei:2026-05-21;15413
15412 2026 2026-05-20 urn:lex:br:federal:lei:2026-05-20;15412
15411 2026 2026-05-20 urn:lex:br:federal:lei:2026-05-20;15411
15410 2026 2026-05-20 urn:lex:br:federal:lei:2026-05-20;15410
15409 2026 2026-05-20 urn:lex:br:federal:lei:2026-05-20;15409
15408 2026 2026-05-14 urn:lex:br:federal:lei:2026-05-14;15408


## 4. Separacao em lotes

O manifest e particionado para dar previsibilidade a execucao, facilitar retomada e evitar perda de progresso em sessoes interrompidas. Os lotes tambem funcionam como trilha de auditoria: e possivel verificar exatamente quais itens foram planejados para coleta antes de iniciar downloads de texto integral.


In [ ]:
batches_summary = criar_lotes_manifest()
print(json.dumps(batches_summary, ensure_ascii=False, indent=2))


{
  "tamanho_lote": 100,
  "total_leis": 7770,
  "total_lotes": 78,
  "primeiro_lote": "/content/corpus_completo_pgvector_local/data/batches/manifest_lote_0001.jsonl",
  "ultimo_lote": "/content/corpus_completo_pgvector_local/data/batches/manifest_lote_0078.jsonl"
}


### Checagem dos lotes

Confere se a soma dos lotes preserva todos os itens do manifest. A etapa falha se houver perda de item durante a particionamento, porque isso comprometeria a cobertura antes mesmo da coleta.


In [ ]:
lote_paths = sorted(BATCHES_DIR.glob("manifest_lote_*.jsonl"))
assert lote_paths
total_lotes = sum(len(read_jsonl(path)) for path in lote_paths)
assert total_lotes == len(read_jsonl(PATHS["manifest"]))
print("Lotes aprovados:", len(lote_paths), "itens:", total_lotes)


Lotes aprovados: 78 itens: 7770


## 5. Coleta do inteiro teor com checkpoint

Para cada lei do manifest, a rotina consulta metadados oficiais, resolve a URL do inteiro teor, baixa o HTML, normaliza o texto, grava a norma integral e cria chunks juridicos. O checkpoint registra status por item para permitir retomada sem duplicar arquivos ja salvos.

O contrato desta etapa e simples: cada URN selecionada precisa terminar com uma norma integral e pelo menos um chunk. Erros de rede, URL indisponivel ou texto insuficiente ficam registrados em `falhas_coleta.jsonl` e sao avaliados na checagem de cobertura.


In [ ]:
coleta_summary = processar_manifest(limite=LIMITE_COLETA, pular_salvos=True)
print(json.dumps(coleta_summary, ensure_ascii=False, indent=2))


OK 1: Lei 15417/2026 - chunks 3
OK 2: Lei 15416/2026 - chunks 5
OK 3: Lei 15415/2026 - chunks 3
OK 4: Lei 15414/2026 - chunks 4
OK 5: Lei 15413/2026 - chunks 4
OK 6: Lei 15412/2026 - chunks 3
OK 7: Lei 15411/2026 - chunks 3
OK 8: Lei 15410/2026 - chunks 5
OK 9: Lei 15409/2026 - chunks 7
OK 10: Lei 15408/2026 - chunks 4
OK 11: Lei 15407/2026 - chunks 5
OK 12: Lei 15406/2026 - chunks 3
OK 13: Lei 15405/2026 - chunks 3
OK 14: Lei 15404/2026 - chunks 6
OK 15: Lei 15403/2026 - chunks 5
OK 16: Lei 15402/2026 - chunks 4
OK 17: Lei 15401/2026 - chunks 7
OK 18: Lei 15400/2026 - chunks 3
OK 19: Lei 15399/2026 - chunks 13
OK 20: Lei 15398/2026 - chunks 19
OK 21: Lei 15397/2026 - chunks 5
OK 22: Lei 15396/2026 - chunks 15
OK 23: Lei 15395/2026 - chunks 28
OK 24: Lei 15394/2026 - chunks 3
OK 25: Lei 15392/2026 - chunks 9
OK 26: Lei 15391/2026 - chunks 22
OK 27: Lei 15390/2026 - chunks 6
OK 28: Lei 15389/2026 - chunks 3
OK 29: Lei 15388/2026 - chunks 38
OK 30: Lei 15387/2026 - chunks 3
OK 31: Lei 15

### Checagem de cobertura da coleta

Compara o manifest selecionado com normas e chunks gravados. Se alguma URN ficar sem norma, sem chunk, duplicada ou com falha registrada, a execucao e interrompida. Esta e a trava que impede o corpus completo de avancar com dados faltantes.


In [ ]:
manifest = read_jsonl(PATHS["manifest"])
normas = read_jsonl(PATHS["normas"])
chunks = read_jsonl(PATHS["chunks"])
falhas_coleta = read_jsonl(PATHS["falhas"])

# Cobertura e medida por URN porque ela e a chave estavel entre manifest,
# norma integral, chunks e banco relacional.
manifest_urns = {item.get("urn") for item in manifest if item.get("urn")}
normas_urns = {item.get("urn") for item in normas if item.get("urn")}
chunks_urns = {item.get("urn") for item in chunks if item.get("urn")}
falhas_urns = {item.get("urn") for item in falhas_coleta if item.get("urn")}

normas_por_urn = Counter(item.get("urn") for item in normas if item.get("urn"))
chunks_por_urn = Counter(item.get("urn") for item in chunks if item.get("urn"))

urns_sem_norma = sorted(manifest_urns - normas_urns)
urns_sem_chunk = sorted(manifest_urns - chunks_urns)
urns_com_falha = sorted(falhas_urns)
urns_norma_duplicada = sorted(urn for urn, total in normas_por_urn.items() if total > 1)
urns_chunk_ausente = sorted(urn for urn in manifest_urns if chunks_por_urn.get(urn, 0) == 0)

# O relatorio guarda amostras das lacunas para diagnostico sem poluir a tela.
cobertura_coleta = {
    "modo": NOTEBOOK_MODE,
    "manifest_total": len(manifest_urns),
    "normas_unicas": len(normas_urns),
    "chunks_urns_unicas": len(chunks_urns),
    "falhas": len(falhas_coleta),
    "processadas_nesta_execucao": coleta_summary.get("processadas"),
    "puladas_por_checkpoint": coleta_summary.get("puladas"),
    "urns_sem_norma": urns_sem_norma[:20],
    "urns_sem_chunk": urns_sem_chunk[:20],
    "urns_com_falha": urns_com_falha[:20],
    "urns_norma_duplicada": urns_norma_duplicada[:20],
    "aprovado": (
        len(manifest_urns) > 0
        and not falhas_coleta
        and not urns_sem_norma
        and not urns_sem_chunk
        and not urns_chunk_ausente
        and not urns_norma_duplicada
    ),
}
PATHS["cobertura_json"].write_text(
    json.dumps(cobertura_coleta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(cobertura_coleta, ensure_ascii=False, indent=2))

# No completo, estes asserts impedem seguir para banco/vetores com corpus incompleto.
assert not falhas_coleta, falhas_coleta[:5]
assert not urns_sem_norma, urns_sem_norma[:20]
assert not urns_sem_chunk, urns_sem_chunk[:20]
assert not urns_chunk_ausente, urns_chunk_ausente[:20]
assert not urns_norma_duplicada, urns_norma_duplicada[:20]
assert cobertura_coleta["aprovado"], cobertura_coleta


### Amostra tecnica da coleta

Exibe uma previa da norma integral e do primeiro chunk para conferir rapidamente formato, fonte e rastreabilidade. A saida deve mostrar titulo normativo, URL oficial, texto legivel e primeiro bloco de recuperacao associado a mesma URN.


In [ ]:
normas = read_jsonl(PATHS["normas"])
chunks = read_jsonl(PATHS["chunks"])
assert normas, "Nenhuma norma salva."
assert chunks, "Nenhum chunk salvo."
primeira_norma = normas[0]
chunks_primeira = [chunk for chunk in chunks if chunk.get("urn") == primeira_norma.get("urn")]
assert chunks_primeira
print("Norma:", primeira_norma["titulo_norma"])
print("URL:", primeira_norma["url_origem"])
print("\n--- Preview texto integral ---")
print(texto_console(primeira_norma["texto_integral"][:1800]))
print("\n--- Preview primeiro chunk ---")
print(texto_console(chunks_primeira[0]["texto_chunk"][:1200]))


## 6. Inspecao do inteiro teor

Salva um arquivo de conferencia com metadados, contagem de caracteres, quantidade de chunks e texto integral formatado da norma selecionada para verificacao. O objetivo e permitir leitura direta do resultado tratado sem depender de banco, vetores ou interface de consulta.

O arquivo gerado fica em `reports/` e inclui a URN, titulo, fonte, URL, tamanho do texto e quantidade de chunks vinculados. Se a norma de referencia nao estiver na amostra, o notebook usa a primeira norma processada.


In [ ]:
normas = read_jsonl(PATHS["normas"])
chunks = read_jsonl(PATHS["chunks"])
assert normas, "Nenhuma norma salva para inspecao."

norma_validacao = next((norma for norma in normas if norma.get("urn") == URN_VALIDACAO_TEXTO_INTEGRAL), None)
if norma_validacao is None:
    norma_validacao = normas[0]

chunks_validacao = [chunk for chunk in chunks if chunk.get("urn") == norma_validacao.get("urn")]
numero_saida = str(norma_validacao.get("numero") or "sem_numero").replace(".", "")
ano_saida = str(norma_validacao.get("ano") or "sem_ano")
arquivo_validacao = REPORTS_DIR / f"texto_integral_validacao_{numero_saida}_{ano_saida}.txt"
conteudo_validacao = "\n".join([
    f"URN: {norma_validacao.get('urn')}",
    f"Titulo: {norma_validacao.get('titulo_norma')}",
    f"Fonte: {norma_validacao.get('fonte_preferida')}",
    f"URL: {norma_validacao.get('url_origem')}",
    f"Caracteres: {len(norma_validacao.get('texto_integral') or '')}",
    f"Chunks vinculados: {len(chunks_validacao)}",
    "",
    "----- TEXTO INTEGRAL FORMATADO -----",
    "",
    norma_validacao.get("texto_integral") or "",
])
arquivo_validacao.write_text(conteudo_validacao, encoding="utf-8")
print("Arquivo com texto integral:", arquivo_validacao)
print(texto_console(conteudo_validacao[:12000]))


## 7. Validacao textual, schema e integridade

Verifica campos obrigatorios, unicidade, hashes, vinculo entre norma integral e chunks, ausencia de texto vazio e padrao minimo de limpeza. Esta validacao cobre consistencia dos arquivos JSONL antes de qualquer carga no PostgreSQL.

A etapa grava relatorio JSON e Markdown em `reports/`. O JSON serve para automacao; o Markdown facilita leitura rapida dos totais e problemas encontrados.


In [ ]:
validacao = salvar_relatorio_validacao()
print(json.dumps(validacao["result"]["summary"], ensure_ascii=False, indent=2))
print("Relatorio JSON:", validacao["json_path"])
print("Relatorio Markdown:", validacao["md_path"])


### Checagem da validacao textual

Interrompe a execucao se a camada textual nao estiver consistente. A etapa seguinte so deve rodar quando normas e chunks estiverem vinculados, sem duplicidade e com texto utilizavel.


In [ ]:
assert validacao["result"]["summary"]["aprovado"], validacao["result"]["problems"][:5]
print("Validacao textual aprovada.")


## 8. Auditoria fina do padrao textual

Complementa a validacao estrutural com verificacoes de acabamento: titulo, marcador de artigo, ruido de topo, ementa fragmentada e caracteres de substituicao. A auditoria e focada na qualidade de leitura e na estabilidade dos chunks que serao vetorizados.

Esta etapa nao altera os dados. Ela apenas mede se o tratamento textual manteve um formato adequado para recuperacao semantica e para consulta posterior no banco.


In [ ]:
auditoria_fina = auditar_padrao_fino_processado()
print(json.dumps(auditoria_fina["result"]["summary"], ensure_ascii=False, indent=2))
print("Relatorio JSON:", auditoria_fina["json_path"])
print("Relatorio Markdown:", auditoria_fina["md_path"])


### Checagem da auditoria fina

Confirma que a amostra processada nao contem os problemas textuais monitorados. Caso falhe, o relatorio aponta as URNs afetadas e as metricas que motivaram a reprovacao.


In [ ]:
assert auditoria_fina["result"]["summary"]["aprovado"], auditoria_fina["result"]["issues"][:5]
print("Auditoria fina aprovada.")


## 9. Preparar PostgreSQL e pgvector

Garante disponibilidade do PostgreSQL, cria o banco quando necessario e habilita a extensao `vector`. Em Colab, o preparo ocorre na propria sessao; em maquina local, o notebook pode usar Docker ou um PostgreSQL ja configurado.

Esta etapa e o ponto de transicao entre arquivos processados e persistencia relacional. A configuracao usa variaveis de ambiente padronizadas para permitir troca de host, porta, banco e credenciais sem alterar as celulas seguintes.


In [ ]:
# Dependencias de banco ficam nesta etapa para permitir testar a captura sem PostgreSQL.
garantir_pacote("psycopg2-binary", "psycopg2")
garantir_pacote("pgvector")
garantir_pacote("sentence-transformers", "sentence_transformers")

import psycopg2
from psycopg2 import sql
from psycopg2.extras import Json, execute_values


def executar_comando(cmd: list[str] | str, shell: bool = False, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, shell=shell, check=check, text=True, capture_output=True)


def comando_disponivel(nome: str) -> bool:
    return shutil.which(nome) is not None


def preparar_postgres_colab() -> None:
    if not EH_COLAB or not PREPARAR_POSTGRES_COLAB:
        return

    # O Colab nao usa Docker; a extensao vector e compilada na propria sessao.
    executar_comando("apt-get update -qq", shell=True)
    executar_comando(
        "apt-get install -y -qq postgresql postgresql-contrib postgresql-server-dev-all build-essential git",
        shell=True,
    )
    control_check = executar_comando("find /usr/share/postgresql -name vector.control | head -n 1", shell=True, check=False)
    if not control_check.stdout.strip():
        executar_comando("rm -rf /tmp/pgvector && git clone --depth 1 https://github.com/pgvector/pgvector.git /tmp/pgvector", shell=True)
        executar_comando("cd /tmp/pgvector && make -s && make install -s", shell=True)
    executar_comando("service postgresql start", shell=True)
    executar_comando(
        f"sudo -u postgres psql -c \"ALTER USER postgres PASSWORD '{DB_PASSWORD}';\"",
        shell=True,
    )


def preparar_postgres_docker() -> None:
    if EH_COLAB or not USAR_DOCKER_LOCAL:
        return
    if not comando_disponivel("docker"):
        print("Docker nao encontrado. Usando PostgreSQL ja disponivel em DB_HOST/DB_PORT.")
        return

    # Em ambiente local, Docker so e usado quando o daemon esta realmente ativo.
    docker_info = executar_comando(["docker", "info"], check=False)
    if docker_info.returncode != 0:
        raise RuntimeError(
            "Docker encontrado, mas o daemon nao respondeu. Em execucao local, "
            "inicie o Docker Desktop ou configure um PostgreSQL existente via "
            "POSTGRES_HOST/POSTGRES_PORT. Em Colab, esta etapa usa PostgreSQL "
            "instalado na propria sessao e nao depende de Docker."
        )

    ps = executar_comando(["docker", "ps", "-a", "--format", "{{.Names}}"], check=False)
    if ps.returncode != 0:
        raise RuntimeError(f"Nao foi possivel listar containers Docker: {ps.stderr.strip()}")

    nomes = set(ps.stdout.splitlines())
    if DB_CONTAINER_NAME not in nomes:
        try:
            executar_comando([
                "docker",
                "run",
                "-d",
                "--name",
                DB_CONTAINER_NAME,
                "-e",
                f"POSTGRES_DB={DB_NAME}",
                "-e",
                f"POSTGRES_USER={DB_USER}",
                "-e",
                f"POSTGRES_PASSWORD={DB_PASSWORD}",
                "-p",
                f"{DB_PORT}:5432",
                "pgvector/pgvector:pg16",
            ])
        except subprocess.CalledProcessError as exc:
            detalhes = (exc.stderr or exc.stdout or str(exc)).strip()
            raise RuntimeError(f"Falha ao iniciar container PostgreSQL pgvector: {detalhes}") from exc
    else:
        executar_comando(["docker", "start", DB_CONTAINER_NAME], check=False)


def conectar(dbname: str | None = None):
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=dbname or DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )


def aguardar_postgres(timeout_seconds: int = 90) -> None:
    inicio = time.time()
    ultimo_erro: Exception | None = None
    while time.time() - inicio < timeout_seconds:
        try:
            conn = conectar("postgres")
            conn.close()
            return
        except Exception as exc:
            ultimo_erro = exc
            time.sleep(2)
    raise RuntimeError(f"PostgreSQL nao respondeu dentro do prazo: {ultimo_erro}")


def garantir_database() -> None:
    conn = conectar("postgres")
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
            if cur.fetchone() is None:
                # CREATE DATABASE nao aceita parametro posicional para identificador.
                cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB_NAME)))
    finally:
        conn.close()


preparar_postgres_colab()
preparar_postgres_docker()
aguardar_postgres()
garantir_database()

print(json.dumps({
    "postgres_ok": True,
    "host": DB_HOST,
    "port": DB_PORT,
    "database": DB_NAME,
    "usuario": DB_USER,
    "colab": EH_COLAB,
    "docker_local": USAR_DOCKER_LOCAL,
}, ensure_ascii=False, indent=2))


### Checagem do PostgreSQL

Confirma a versao do servidor e a disponibilidade da extensao pgvector. Sem essa extensao, o schema vetorial nao pode ser criado e a execucao deve parar antes da carga.


In [ ]:
with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        postgres_version = cur.fetchone()[0]
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute("SELECT extname FROM pg_extension WHERE extname = 'vector';")
        vector_extension = cur.fetchone()[0]
assert vector_extension == "vector"
print("PostgreSQL:", postgres_version)
print("Extensao:", vector_extension)


## 10. Criar schema relacional e vetorial

O banco usa duas tabelas principais: `normas_integras` para o texto completo e `chunks_juridicos` para a camada de recuperacao. O campo `embedding` usa `vector(384)`, alinhado ao modelo configurado.

A separacao entre norma e chunk preserva o inteiro teor sem misturar com a camada de busca. Cada chunk referencia uma URN existente, guarda ordem, texto original do bloco, texto contextualizado e metadados em JSONB.


In [ ]:
# `normas_integras` preserva o texto completo; `chunks_juridicos` e a camada de busca.
# A dimensao do campo vector precisa bater com o modelo carregado na etapa de embeddings.
CREATE_SCHEMA_SQL = f"""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS normas_integras (
    id SERIAL PRIMARY KEY,
    urn TEXT UNIQUE NOT NULL,
    tipo_norma TEXT,
    numero TEXT,
    ano TEXT,
    data_assinatura TEXT,
    titulo_norma TEXT,
    ementa TEXT,
    fonte_preferida TEXT,
    url_origem TEXT,
    texto_integral TEXT NOT NULL,
    quantidade_caracteres INTEGER NOT NULL,
    hash_texto TEXT,
    coletado_em TEXT,
    pipeline_version TEXT,
    json_canonico JSONB NOT NULL,
    metadados_senado JSONB,
    raw_senado JSONB,
    erros_fontes JSONB
);

CREATE TABLE IF NOT EXISTS chunks_juridicos (
    id SERIAL PRIMARY KEY,
    chunk_id TEXT UNIQUE NOT NULL,
    urn TEXT NOT NULL REFERENCES normas_integras(urn) ON DELETE CASCADE,
    tipo_norma TEXT,
    numero TEXT,
    ano TEXT,
    titulo_norma TEXT,
    ementa TEXT,
    fonte TEXT,
    tipo_bloco TEXT,
    artigo TEXT,
    paragrafo TEXT,
    inciso TEXT,
    hierarquia_titulo TEXT,
    hierarquia_capitulo TEXT,
    hierarquia_secao TEXT,
    hierarquia_subsecao TEXT,
    texto_chunk TEXT NOT NULL,
    texto_contextualizado TEXT,
    url_origem TEXT,
    coletado_em TEXT,
    pipeline_version TEXT,
    ordem_chunk INTEGER NOT NULL,
    metadata_json JSONB NOT NULL,
    embedding vector({EMBEDDING_DIMENSION}),
    embedding_modelo TEXT,
    embedding_campo_textual TEXT,
    embedding_normalizado BOOLEAN,
    embedding_atualizado_em TIMESTAMPTZ
);

CREATE INDEX IF NOT EXISTS idx_chunks_urn ON chunks_juridicos(urn);
CREATE INDEX IF NOT EXISTS idx_chunks_artigo ON chunks_juridicos(artigo);
CREATE INDEX IF NOT EXISTS idx_chunks_ordem ON chunks_juridicos(urn, ordem_chunk);
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_SCHEMA_SQL)
        if RESETAR_TABELAS:
            # Reset usado na amostra para manter contagens deterministicas entre execucoes.
            cur.execute("TRUNCATE TABLE chunks_juridicos, normas_integras RESTART IDENTITY CASCADE;")

print(json.dumps({
    "schema_criado": True,
    "pgvector_dimension": EMBEDDING_DIMENSION,
    "resetar_tabelas": RESETAR_TABELAS,
}, ensure_ascii=False, indent=2))


### Checagem do schema

Confere se as tabelas foram criadas no banco configurado. Esta checagem confirma que a extensao `vector` e as tabelas de texto integral e chunks estao prontas para receber carga.


In [ ]:
with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;"
        )
        tabelas = [row[0] for row in cur.fetchall()]
assert "normas_integras" in tabelas
assert "chunks_juridicos" in tabelas
print("Tabelas:", tabelas)


## 11. Carregar normas e chunks no PostgreSQL

Os arquivos JSONL produzidos pela captura sao materializados no banco. A carga usa upsert por `urn` e `chunk_id`, preservando rastreabilidade e permitindo reexecucao.

O banco passa a ser a fonte de trabalho para vetorizacao e busca. As contagens retornadas nesta etapa precisam ser compativeis com os arquivos processados, considerando que o completo pode acumular registros validos de execucoes anteriores. Se um chunk ja existente tiver texto alterado em nova carga, o embedding correspondente e limpo para ser recalculado; isso evita manter vetor antigo para texto atualizado.


In [ ]:
normas_para_carga = read_jsonl(PATHS["normas"])
chunks_para_carga = read_jsonl(PATHS["chunks"])
assert normas_para_carga, "Nenhuma norma encontrada para carga."
assert chunks_para_carga, "Nenhum chunk encontrado para carga."


def upsert_normas(normas: list[dict[str, Any]]) -> int:
    # Upsert por URN permite retomar o processamento sem duplicar norma integral.
    rows = []
    for norma in normas:
        rows.append((
            norma.get("urn"),
            norma.get("tipo_norma"),
            norma.get("numero"),
            norma.get("ano"),
            norma.get("data_assinatura"),
            norma.get("titulo_norma"),
            norma.get("ementa"),
            norma.get("fonte_preferida"),
            norma.get("url_origem"),
            norma.get("texto_integral"),
            int(norma.get("quantidade_caracteres") or len(norma.get("texto_integral") or "")),
            norma.get("hash_texto"),
            norma.get("coletado_em"),
            norma.get("pipeline_version"),
            Json(norma.get("json_canonico") or norma),
            Json(norma.get("metadados_senado")),
            Json(norma.get("raw_senado")),
            Json(norma.get("erros_fontes")),
        ))
    query = """
        INSERT INTO normas_integras (
            urn, tipo_norma, numero, ano, data_assinatura, titulo_norma,
            ementa, fonte_preferida, url_origem, texto_integral,
            quantidade_caracteres, hash_texto, coletado_em, pipeline_version,
            json_canonico, metadados_senado, raw_senado, erros_fontes
        ) VALUES %s
        ON CONFLICT (urn) DO UPDATE SET
            tipo_norma = EXCLUDED.tipo_norma,
            numero = EXCLUDED.numero,
            ano = EXCLUDED.ano,
            data_assinatura = EXCLUDED.data_assinatura,
            titulo_norma = EXCLUDED.titulo_norma,
            ementa = EXCLUDED.ementa,
            fonte_preferida = EXCLUDED.fonte_preferida,
            url_origem = EXCLUDED.url_origem,
            texto_integral = EXCLUDED.texto_integral,
            quantidade_caracteres = EXCLUDED.quantidade_caracteres,
            hash_texto = EXCLUDED.hash_texto,
            coletado_em = EXCLUDED.coletado_em,
            pipeline_version = EXCLUDED.pipeline_version,
            json_canonico = EXCLUDED.json_canonico,
            metadados_senado = EXCLUDED.metadados_senado,
            raw_senado = EXCLUDED.raw_senado,
            erros_fontes = EXCLUDED.erros_fontes;
    """
    with conectar() as conn:
        with conn.cursor() as cur:
            execute_values(cur, query, rows, page_size=100)
    return len(rows)


def upsert_chunks(chunks: list[dict[str, Any]]) -> int:
    # `chunk_id` e deterministico, entao reexecutar a carga atualiza o registro existente.
    rows = []
    for chunk in chunks:
        rows.append((
            chunk.get("chunk_id"),
            chunk.get("urn"),
            chunk.get("tipo_norma"),
            chunk.get("numero"),
            chunk.get("ano"),
            chunk.get("titulo_norma"),
            chunk.get("ementa"),
            chunk.get("fonte"),
            chunk.get("tipo_bloco"),
            chunk.get("artigo"),
            chunk.get("paragrafo"),
            chunk.get("inciso"),
            chunk.get("hierarquia_titulo"),
            chunk.get("hierarquia_capitulo"),
            chunk.get("hierarquia_secao"),
            chunk.get("hierarquia_subsecao"),
            chunk.get("texto_chunk"),
            chunk.get("texto_contextualizado"),
            chunk.get("url_origem"),
            chunk.get("coletado_em"),
            chunk.get("pipeline_version"),
            int(chunk.get("ordem_chunk") or 0),
            Json(chunk.get("metadata_json") or chunk),
        ))
    query = """
        INSERT INTO chunks_juridicos (
            chunk_id, urn, tipo_norma, numero, ano, titulo_norma, ementa,
            fonte, tipo_bloco, artigo, paragrafo, inciso, hierarquia_titulo,
            hierarquia_capitulo, hierarquia_secao, hierarquia_subsecao,
            texto_chunk, texto_contextualizado, url_origem, coletado_em,
            pipeline_version, ordem_chunk, metadata_json
        ) VALUES %s
    ON CONFLICT (chunk_id) DO UPDATE SET
        urn = EXCLUDED.urn,
        tipo_norma = EXCLUDED.tipo_norma,
            numero = EXCLUDED.numero,
            ano = EXCLUDED.ano,
            titulo_norma = EXCLUDED.titulo_norma,
            ementa = EXCLUDED.ementa,
            fonte = EXCLUDED.fonte,
            tipo_bloco = EXCLUDED.tipo_bloco,
            artigo = EXCLUDED.artigo,
            paragrafo = EXCLUDED.paragrafo,
            inciso = EXCLUDED.inciso,
            hierarquia_titulo = EXCLUDED.hierarquia_titulo,
            hierarquia_capitulo = EXCLUDED.hierarquia_capitulo,
            hierarquia_secao = EXCLUDED.hierarquia_secao,
            hierarquia_subsecao = EXCLUDED.hierarquia_subsecao,
            texto_chunk = EXCLUDED.texto_chunk,
            texto_contextualizado = EXCLUDED.texto_contextualizado,
            url_origem = EXCLUDED.url_origem,
        coletado_em = EXCLUDED.coletado_em,
        pipeline_version = EXCLUDED.pipeline_version,
        ordem_chunk = EXCLUDED.ordem_chunk,
        metadata_json = EXCLUDED.metadata_json,
        embedding = CASE
            WHEN chunks_juridicos.texto_chunk IS DISTINCT FROM EXCLUDED.texto_chunk
              OR chunks_juridicos.texto_contextualizado IS DISTINCT FROM EXCLUDED.texto_contextualizado
            THEN NULL
            ELSE chunks_juridicos.embedding
        END,
        embedding_modelo = CASE
            WHEN chunks_juridicos.texto_chunk IS DISTINCT FROM EXCLUDED.texto_chunk
              OR chunks_juridicos.texto_contextualizado IS DISTINCT FROM EXCLUDED.texto_contextualizado
            THEN NULL
            ELSE chunks_juridicos.embedding_modelo
        END,
        embedding_campo_textual = CASE
            WHEN chunks_juridicos.texto_chunk IS DISTINCT FROM EXCLUDED.texto_chunk
              OR chunks_juridicos.texto_contextualizado IS DISTINCT FROM EXCLUDED.texto_contextualizado
            THEN NULL
            ELSE chunks_juridicos.embedding_campo_textual
        END,
        embedding_normalizado = CASE
            WHEN chunks_juridicos.texto_chunk IS DISTINCT FROM EXCLUDED.texto_chunk
              OR chunks_juridicos.texto_contextualizado IS DISTINCT FROM EXCLUDED.texto_contextualizado
            THEN NULL
            ELSE chunks_juridicos.embedding_normalizado
        END,
        embedding_atualizado_em = CASE
            WHEN chunks_juridicos.texto_chunk IS DISTINCT FROM EXCLUDED.texto_chunk
              OR chunks_juridicos.texto_contextualizado IS DISTINCT FROM EXCLUDED.texto_contextualizado
            THEN NULL
            ELSE chunks_juridicos.embedding_atualizado_em
        END;
"""
    with conectar() as conn:
        with conn.cursor() as cur:
            execute_values(cur, query, rows, page_size=500)
    return len(rows)


normas_carregadas = upsert_normas(normas_para_carga)
chunks_carregados = upsert_chunks(chunks_para_carga)

# As contagens consultadas no banco fecham a etapa de persistencia antes da vetorizacao.
with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM normas_integras;")
        total_normas_banco = cur.fetchone()[0]
        cur.execute("SELECT count(*) FROM chunks_juridicos;")
        total_chunks_banco = cur.fetchone()[0]

carga_summary = {
    "normas_lidas": len(normas_para_carga),
    "chunks_lidos": len(chunks_para_carga),
    "normas_carregadas": normas_carregadas,
    "chunks_carregados": chunks_carregados,
    "total_normas_banco": total_normas_banco,
    "total_chunks_banco": total_chunks_banco,
}
print(json.dumps(carga_summary, ensure_ascii=False, indent=2))


### Checagem da carga

Confirma se as contagens do banco sao suficientes para cobrir os arquivos processados. A etapa falha se nao houver normas ou chunks para carregar.


In [ ]:
assert carga_summary["total_normas_banco"] >= carga_summary["normas_lidas"]
assert carga_summary["total_chunks_banco"] >= carga_summary["chunks_lidos"]
assert carga_summary["normas_lidas"] > 0
assert carga_summary["chunks_lidos"] > 0
print("Carga aprovada.")


## 12. Carregar modelo de embeddings

O modelo transforma texto em vetores numericos. Para o padrao definido, cada vetor possui 384 dimensoes e e normalizado para busca por distancia de cosseno.

O carregamento aceita duas fontes: `EMBEDDING_MODEL_PATH`, quando houver um diretorio local explicito, ou `EMBEDDING_MODEL`, quando o ambiente precisa resolver o modelo por nome. Antes de processar todos os chunks, a celula gera um vetor sentinela para confirmar dimensao e normalizacao.


In [ ]:
from sentence_transformers import SentenceTransformer


# O modelo E5 usa prefixos distintos para documentos e consultas.
def texto_passage_e5(texto: str) -> str:
    return "passage: " + texto.strip()


def texto_query_e5(pergunta: str) -> str:
    return "query: " + pergunta.strip()


def texto_para_embedding_chunk(row: dict[str, Any]) -> tuple[str, str]:
    # Prioriza o texto contextualizado porque ele carrega titulo, ementa e artigo.
    texto_contextualizado = (row.get("texto_contextualizado") or "").strip()
    if texto_contextualizado:
        return texto_contextualizado, "texto_contextualizado"
    texto_chunk = (row.get("texto_chunk") or "").strip()
    if texto_chunk:
        return texto_chunk, "texto_chunk"
    raise ValueError(f"Chunk sem texto para embedding: {row.get('chunk_id')}")


def carregar_modelo_embedding():
    if EMBEDDING_MODEL_PATH:
        origem = Path(EMBEDDING_MODEL_PATH)
        if not origem.exists():
            raise FileNotFoundError(
                "EMBEDDING_MODEL_PATH foi informado, mas o diretorio nao existe: "
                f"{origem}. Ajuste o caminho ou deixe EMBEDDING_MODEL_PATH vazio "
                "para carregar o modelo padrao por nome."
            )
        if not origem.is_dir():
            raise NotADirectoryError(
                "EMBEDDING_MODEL_PATH precisa apontar para um diretorio de modelo: "
                f"{origem}"
            )
        # Caminho local explicito e tratado como fonte fechada e reproduzivel.
        return SentenceTransformer(str(origem), local_files_only=True)

    if not EMBEDDING_MODEL:
        raise ValueError("EMBEDDING_MODEL nao pode ficar vazio quando EMBEDDING_MODEL_PATH nao foi informado.")

    try:
        # Em Colab limpo, o modelo padrao e resolvido por nome e fica em cache local.
        return SentenceTransformer(
            EMBEDDING_MODEL,
            cache_folder=EMBEDDING_CACHE_DIR or None,
            local_files_only=EMBEDDING_LOCAL_FILES_ONLY,
        )
    except OSError as exc:
        if EMBEDDING_LOCAL_FILES_ONLY:
            raise FileNotFoundError(
                "Modelo nao encontrado no cache local. Defina EMBEDDING_LOCAL_FILES_ONLY=false "
                "para permitir o carregamento por nome, ou configure EMBEDDING_MODEL_PATH "
                "com um diretorio de modelo ja disponivel."
            ) from exc
        raise


def vetor_para_pgvector(vetor: list[float]) -> str:
    return "[" + ",".join(f"{float(value):.10f}" for value in vetor) + "]"


modelo_embedding = carregar_modelo_embedding()

# Vetor sentinela: confirma dimensao e normalizacao antes de gravar no banco.
vetor_teste = modelo_embedding.encode(
    texto_query_e5("teste de busca juridica"),
    normalize_embeddings=EMBEDDING_NORMALIZE,
)
vetor_teste = [float(value) for value in vetor_teste.tolist()]
norma_teste = math.sqrt(sum(value * value for value in vetor_teste))
assert len(vetor_teste) == EMBEDDING_DIMENSION, len(vetor_teste)
assert 0.99 < norma_teste < 1.01, norma_teste

print(json.dumps({
    "modelo_embedding": EMBEDDING_MODEL_LABEL,
    "modelo_id": EMBEDDING_MODEL,
    "modelo_path": EMBEDDING_MODEL_PATH or None,
    "modelo_source": EMBEDDING_MODEL_SOURCE,
    "cache_dir": EMBEDDING_CACHE_DIR or None,
    "dimensao": len(vetor_teste),
    "norma_vetor_teste": norma_teste,
    "local_files_only": True if EMBEDDING_MODEL_PATH else EMBEDDING_LOCAL_FILES_ONLY,
}, ensure_ascii=False, indent=2))


### Checagem do modelo

Confirma dimensao e normalizacao antes de gravar vetores no banco. Se a dimensao nao for 384, o schema e o modelo estao desalinhados e a execucao deve ser interrompida.


In [ ]:
assert len(vetor_teste) == EMBEDDING_DIMENSION
assert 0.99 < norma_teste < 1.01
print("Modelo aprovado.")


## 13. Vetorizar chunks e gravar em pgvector

A vetorizacao percorre chunks pendentes em lotes. O texto preferencial e `texto_contextualizado`; se estiver ausente, usa `texto_chunk`.

O processo e incremental: apenas chunks sem embedding sao selecionados. Isso permite retomada depois de interrupcao sem recalcular vetores ja gravados. Cada vetor gravado registra modelo, campo textual usado, normalizacao e timestamp.


In [ ]:
def listar_chunks_pendentes(limite: int) -> list[dict[str, Any]]:
    # Processa apenas registros sem embedding para permitir retomada incremental.
    with conectar() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT chunk_id, texto_chunk, texto_contextualizado
                FROM chunks_juridicos
                WHERE embedding IS NULL
                ORDER BY urn ASC, ordem_chunk ASC
                LIMIT %s;
                """,
                (limite,),
            )
            rows = cur.fetchall()
    return [
        {"chunk_id": row[0], "texto_chunk": row[1], "texto_contextualizado": row[2]}
        for row in rows
    ]


def atualizar_embeddings(payloads: list[tuple[str, str, str, bool, str]]) -> None:
    # Atualizacao em lote reduz round-trips e preserva metadados do vetor gravado.
    query = """
        UPDATE chunks_juridicos AS c
        SET
            embedding = v.embedding::vector,
            embedding_modelo = v.embedding_modelo,
            embedding_campo_textual = v.embedding_campo_textual,
            embedding_normalizado = v.embedding_normalizado,
            embedding_atualizado_em = v.embedding_atualizado_em::timestamptz
        FROM (VALUES %s) AS v(
            chunk_id,
            embedding,
            embedding_modelo,
            embedding_campo_textual,
            embedding_normalizado,
            embedding_atualizado_em
        )
        WHERE c.chunk_id = v.chunk_id;
    """
    with conectar() as conn:
        with conn.cursor() as cur:
            execute_values(cur, query, payloads, page_size=500)


total_processados = 0
inicio_vetorizacao = time.perf_counter()

while True:
    pendentes = listar_chunks_pendentes(EMBEDDING_BATCH_SIZE)
    if not pendentes:
        break

    # O texto enviado ao modelo segue o mesmo formato usado na busca.
    textos_modelo = []
    campos = []
    for row in pendentes:
        texto, campo = texto_para_embedding_chunk(row)
        textos_modelo.append(texto_passage_e5(texto))
        campos.append(campo)

    embeddings = modelo_embedding.encode(
        textos_modelo,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=EMBEDDING_NORMALIZE,
        show_progress_bar=False,
    )
    atualizado_em = datetime.now(timezone.utc).isoformat()
    payloads = []
    for row, campo, embedding in zip(pendentes, campos, embeddings, strict=True):
        vetor = [float(value) for value in embedding.tolist()]
        payloads.append((
            row["chunk_id"],
            vetor_para_pgvector(vetor),
            EMBEDDING_MODEL_LABEL,
            campo,
            EMBEDDING_NORMALIZE,
            atualizado_em,
        ))

    atualizar_embeddings(payloads)
    total_processados += len(payloads)
    print(f"Embeddings gravados nesta execucao: {total_processados}")

duracao_vetorizacao = time.perf_counter() - inicio_vetorizacao

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM chunks_juridicos;")
        total_chunks = cur.fetchone()[0]
        cur.execute("SELECT count(*) FROM chunks_juridicos WHERE embedding IS NOT NULL;")
        chunks_com_embedding = cur.fetchone()[0]

# A etapa so aprova quando todos os chunks carregados possuem vetor.
vectorization_summary = {
    "chunks_total": total_chunks,
    "chunks_com_embedding": chunks_com_embedding,
    "chunks_vetorizados_nesta_execucao": total_processados,
    "embedding_model": EMBEDDING_MODEL_LABEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "embedding_normalized": EMBEDDING_NORMALIZE,
    "duracao_segundos": round(duracao_vetorizacao, 2),
    "aprovado": total_chunks > 0 and chunks_com_embedding == total_chunks,
}
print(json.dumps(vectorization_summary, ensure_ascii=False, indent=2))
assert vectorization_summary["aprovado"], vectorization_summary


### Checagem da vetorizacao

Confirma que todos os chunks carregados possuem embedding gravado. Esta e a barreira antes da busca: nao deve haver chunk elegivel sem vetor.


In [ ]:
assert vectorization_summary["aprovado"], vectorization_summary
assert vectorization_summary["chunks_com_embedding"] == vectorization_summary["chunks_total"]
print("Vetorizacao aprovada.")


## 14. Criar indice vetorial e executar busca

O indice HNSW acelera consultas por similaridade. A busca de verificacao calcula o embedding da pergunta e ordena os chunks pela distancia vetorial.

A consulta usa o mesmo modelo, normalizacao e formato de entrada da vetorizacao. O resultado precisa retornar identificador de chunk, URN, titulo, trecho textual e distancia, provando que a camada pgvector esta operacional.

Esta busca e uma prova de funcionamento da camada vetorial, nao uma avaliacao juridica final. Na amostra, a consulta padrao e concentrada na norma de referencia para tornar a demonstracao deterministica; no corpus completo, a mesma consulta roda sem filtro de URN e mede a recuperacao em toda a base carregada.


In [ ]:
# HNSW acelera consultas por distancia cosseno quando a tabela cresce.
with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE INDEX IF NOT EXISTS idx_chunks_embedding_hnsw_cosine
            ON chunks_juridicos
            USING hnsw (embedding vector_cosine_ops)
            WHERE embedding IS NOT NULL;
            """
        )


def buscar_pgvector(pergunta: str, top_k: int = 5, urn: str | None = None) -> list[dict[str, Any]]:
    # A consulta usa o mesmo modelo e a mesma normalizacao da vetorizacao dos chunks.
    vetor = modelo_embedding.encode(
        texto_query_e5(pergunta),
        normalize_embeddings=EMBEDDING_NORMALIZE,
    )
    vetor_text = vetor_para_pgvector([float(value) for value in vetor.tolist()])

    where = "embedding IS NOT NULL"
    params: list[Any] = [vetor_text]
    if urn:
        # Na amostra, o filtro deixa a verificacao mais deterministica.
        where += " AND urn = %s"
        params.append(urn)
    params.extend([vetor_text, top_k])

    query = f"""
        SELECT
            chunk_id,
            urn,
            titulo_norma,
            artigo,
            texto_chunk,
            url_origem,
            embedding <=> %s::vector AS distancia
        FROM chunks_juridicos
        WHERE {where}
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """
    with conectar() as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            rows = cur.fetchall()

    return [
        {
            "chunk_id": row[0],
            "urn": row[1],
            "titulo_norma": row[2],
            "artigo": row[3],
            "texto_chunk": row[4],
            "url_origem": row[5],
            "distancia": float(row[6]),
        }
        for row in rows
    ]


pergunta_teste = PERGUNTA_TESTE_BUSCA
urn_filtro = URN_REFERENCIA_VALIDACAO if (not RODAR_CORPUS_COMPLETO and URN_REFERENCIA_VALIDACAO) else None
resultados_busca = buscar_pgvector(pergunta_teste, top_k=5, urn=urn_filtro)

assert resultados_busca, "Busca pgvector nao retornou resultados."
primeiro = resultados_busca[0]
assert primeiro["chunk_id"]
assert primeiro["urn"]
assert primeiro["distancia"] >= 0

print("Pergunta:", pergunta_teste)
for indice, item in enumerate(resultados_busca, start=1):
    print("\nResultado", indice)
    print("Distancia:", item["distancia"])
    print("Chunk:", item["chunk_id"])
    print("URN:", item["urn"])
    print("Titulo:", item["titulo_norma"])
    print("Artigo:", item["artigo"])
    print(texto_console((item["texto_chunk"] or "")[:700]))

print("Busca pgvector aprovada.")


### Checagem da busca pgvector

Confirma retorno de resultado com identificador, URN e distancia. A checagem nao avalia resposta juridica final; ela valida a recuperacao vetorial sobre os chunks persistidos.


In [ ]:
assert resultados_busca
assert resultados_busca[0]["chunk_id"]
assert resultados_busca[0]["urn"]
assert resultados_busca[0]["distancia"] >= 0
print("Busca aprovada.")


## 15. Resumo tecnico final

O resumo consolida entradas, saidas, contagens, modelo, dimensao vetorial, validacoes e resultado da busca. O arquivo final em `reports/` registra o estado da execucao e serve como evidencia tecnica de conclusao.

A execucao so e aprovada quando ha normas, chunks, embeddings para todos os chunks e resultado de busca. Se algum desses pontos falhar, o resumo tambem falha.


In [ ]:
# O resumo cruza arquivos, banco e validacoes para registrar o estado final da execucao.
with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM normas_integras;")
        total_normas_final = cur.fetchone()[0]
        cur.execute("SELECT count(*) FROM chunks_juridicos;")
        total_chunks_final = cur.fetchone()[0]
        cur.execute("SELECT count(*) FROM chunks_juridicos WHERE embedding IS NOT NULL;")
        total_embeddings_final = cur.fetchone()[0]

pgvector_summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "modo": NOTEBOOK_MODE,
    "base_dir": str(BASE_DIR),
    "manifest_total_leis": len(read_jsonl(PATHS["manifest"])),
    "normas_processadas_jsonl": len(read_jsonl(PATHS["normas"])),
    "chunks_processados_jsonl": len(read_jsonl(PATHS["chunks"])),
    "normas_postgres": total_normas_final,
    "chunks_postgres": total_chunks_final,
    "chunks_com_embedding": total_embeddings_final,
    "embedding_model": EMBEDDING_MODEL_LABEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "validacao_textual_aprovada": validacao["result"]["summary"]["aprovado"],
    "auditoria_textual_aprovada": auditoria_fina["result"]["summary"]["aprovado"],
    "pergunta_teste_busca": pergunta_teste,
    "busca_pgvector_resultados": len(resultados_busca),
    "busca_pgvector_top1_distancia": resultados_busca[0]["distancia"] if resultados_busca else None,
    "busca_pgvector_top1_chunk_id": resultados_busca[0]["chunk_id"] if resultados_busca else None,
    "publicacao_remota": False,
    "aprovado": (
        total_normas_final > 0
        and total_chunks_final > 0
        and total_embeddings_final == total_chunks_final
        and len(resultados_busca) > 0
    ),
}
PATHS["pgvector_summary"].write_text(
    json.dumps(pgvector_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(pgvector_summary, ensure_ascii=False, indent=2))
assert pgvector_summary["aprovado"], pgvector_summary
print("Notebook finalizado com PostgreSQL + pgvector.")
